# MahimandalSR reviewer setup

These five notebooks contain the dataset preparation and benchmark workflows
used for the paper. Saved outputs were cleared. Submission copies clarify setup,
limit the forward transfer to the published baselines and preserve the reported
data, model, training and metric definitions.

**Dataset.** Use the frozen [reflectance16 release](https://www.kaggle.com/datasets/twilight2002/mahimandal-sr-reflectance16/)
for reproducing benchmark results. Its `paper_release_manifest.csv` has SHA-256
`6bf97403ee40ccf0cbd38b2e6e4932e01973e5d00e211e0acfb1e0b810efc5e7` and 3,424/645/731 train/validation/test pairs. Notebook 01 documents
the original Kaggle collection workflow; rerunning it against current imagery
catalogues is not a substitute for using the frozen release.

**Environment.** Notebooks 02--05 use the recorded DGX workflow: Python 3.10+
and a compatible CUDA-enabled PyTorch/torchvision environment with one assigned
GPU, Git and access to pinned model repositories. Start Jupyter in a private
working directory named `25mcsa19`, the original folder guard. Use that same
working directory for all four notebooks. Runtime dependencies and embedded
runner setup are in the notebook cells. Kaggle download cells need your own
`kaggle.json` in the working directory; keep it outside the submitted files.

**Run order.** 02 is seed 42, 03 adds seeds 43 and 44, 04 evaluates Mahimandal
checkpoints on OLI2MSI, and 05 fits on OLI2MSI and tests on both datasets.
Keep test cells locked until validation decisions are fixed. Optional download
and gallery-export cells are not needed for the numerical results.

**Archived evidence.** The reviewer-access bundle supplies checkpoints,
preflight receipts and per-image scores separately. Saved scores can be used
to verify the paper's numerical summaries without retraining. Use only trusted
checkpoints. These notebooks do not contain imagery, weights or credentials.


# Kaggle: Landsat 8/9 + Sentinel-2 same-day 3x SR pairs inside India

This notebook queries Microsoft Planetary Computer and writes real aligned
`4 x 160 x 160` Landsat RGB+NIR inputs at 30 m with `4 x 480 x 480`
Sentinel-2 RGB+NIR targets at 10 m.

**Kaggle settings:** enable Internet; CPU is sufficient. Run top to bottom.
The collection loop is resumable inside `/kaggle/working`. To resume in a
later Kaggle session, save the previous output as a Kaggle dataset, attach it,
set `RESTORE_FROM`, and rerun from the configuration cell.

The public boundary fallback is suitable for filtering, but it is not an
official Survey of India boundary. Set `INDIA_BOUNDARY_FILE` to an attached
authoritative boundary when required by the release protocol.

In [ ]:
import subprocess, sys

REQUIREMENTS = [
    'pystac-client>=0.8,<1',
    'planetary-computer>=1,<2',
    'shapely>=2,<3',
]
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *REQUIREMENTS],
               check=True)

In [ ]:
"""Landscape metadata and balancing helpers for the India PC collector."""
from __future__ import annotations

import hashlib
from collections import Counter
from typing import Mapping

import numpy as np
import pandas as pd


WORLDCOVER_CLASSES = {
    10: "tree",
    20: "shrubland",
    30: "grassland",
    40: "cropland",
    50: "built_up",
    60: "bare_sparse",
    70: "snow_ice",
    80: "water",
    90: "wetland",
    95: "mangrove",
    100: "moss_lichen",
}
WORLDCOVER_COLUMNS = tuple(
    f"wc2021_{name}_pct" for name in WORLDCOVER_CLASSES.values()
)

# Esri/Impact Observatory Sentinel-2 10 m Annual Land Cover.  Codes 3 and 6
# are unused by the nine-class product; zero is treated as no data.
ESRI_LANDCOVER_CLASSES = {
    1: "water",
    2: "trees",
    4: "flooded_vegetation",
    5: "crops",
    7: "built",
    8: "bare",
    9: "snow_ice",
    10: "clouds",
    11: "rangeland",
}
ESRI_LANDCOVER_COLUMNS = tuple(
    f"esri2025_{name}_pct" for name in ESRI_LANDCOVER_CLASSES.values()
)

LANDSCAPE_CLASSES = (
    "alpine",
    "mountain",
    "urban",
    "desert_bare",
    "forest",
    "cropland",
    "water_wetland",
    "shrub_grass",
    "mixed",
)

# These are collection reserves, not promises. If a rare class cannot satisfy its
# reserve, collection stops below TARGET_PAIRS rather than silently replacing it.
DEFAULT_LANDSCAPE_MINIMUMS = {
    "alpine": 150,
    "mountain": 450,
    "urban": 500,
    "desert_bare": 450,
    "forest": 600,
    "cropland": 800,
    "water_wetland": 300,
    "shrub_grass": 550,
    "mixed": 500,
}

# Hard caps prevent the common classes discovered first by the catalogue scan
# from consuming the release.  The minima sum to 4,300, leaving 500 flexible
# slots while cropland can occupy at most 20.8% of a 4,800-pair release.
DEFAULT_LANDSCAPE_MAXIMUMS = {
    "alpine": 500,
    "mountain": 800,
    "urban": 800,
    "desert_bare": 700,
    "forest": 900,
    "cropland": 1_000,
    "water_wetland": 600,
    "shrub_grass": 800,
    "mixed": 700,
}

GEOGRAPHIC_STRATA = ("extreme_north", "northern_band", "rest_of_india")
DEFAULT_GEOGRAPHIC_MINIMUMS = {
    "extreme_north": 300,
    "northern_band": 500,
    "rest_of_india": 4000,
}

WESTERN_HIMALAYA_STATES = (
    "jammu and kashmir",
    "ladakh",
    "himachal pradesh",
    "uttarakhand",
)
PRIORITY_REGIONS = ("western_himalaya", "rest_of_india")
DEFAULT_PRIORITY_REGION_MINIMUMS = {
    "western_himalaya": 800,
    "rest_of_india": 4000,
}


def geographic_stratum(latitude: float) -> str:
    latitude = float(latitude)
    if latitude >= 33.0:
        return "extreme_north"
    if latitude >= 29.0:
        return "northern_band"
    return "rest_of_india"


def worldcover_percentages(labels: np.ndarray) -> dict[str, float]:
    labels = np.asarray(labels)
    valid = labels != 0
    denominator = int(valid.sum())
    if denominator == 0:
        raise ValueError("WorldCover patch contains no valid labels")
    return {
        name: float((labels[valid] == value).mean() * 100.0)
        for value, name in WORLDCOVER_CLASSES.items()
    }


def esri_landcover_percentages(labels: np.ndarray) -> dict[str, float]:
    """Return annual 2025 class percentages over valid ImageServer pixels."""

    labels = np.asarray(labels)
    valid_codes = np.asarray(tuple(ESRI_LANDCOVER_CLASSES), dtype=labels.dtype)
    valid = np.isin(labels, valid_codes)
    denominator = int(valid.sum())
    if denominator == 0:
        raise ValueError("Esri 2025 land-cover patch contains no valid labels")
    return {
        name: float((labels[valid] == value).mean() * 100.0)
        for value, name in ESRI_LANDCOVER_CLASSES.items()
    }


def terrain_statistics(elevation: np.ndarray, resolution_m: float = 30.0) -> dict[str, float]:
    elevation = np.asarray(elevation, dtype=np.float32)
    valid = np.isfinite(elevation) & (elevation > -500)
    if valid.mean() < 0.95:
        raise ValueError("DEM patch has insufficient valid coverage")
    filled = np.where(valid, elevation, np.nanmedian(elevation[valid]))
    vertical, horizontal = np.gradient(filled, float(resolution_m), float(resolution_m))
    slope = np.degrees(np.arctan(np.hypot(horizontal, vertical)))
    return {
        "elevation_mean_m": float(np.mean(filled)),
        "elevation_p90_m": float(np.percentile(filled, 90)),
        "elevation_max_m": float(np.max(filled)),
        "slope_mean_deg": float(np.mean(slope)),
        "slope_p90_deg": float(np.percentile(slope, 90)),
    }


def classify_landscape(
    cover: Mapping[str, float], terrain: Mapping[str, float]
) -> str:
    """Assign one transparent collection stratum from WorldCover plus DEM."""

    alpine_cover = cover.get("snow_ice", 0.0) + cover.get("moss_lichen", 0.0)
    if (
        alpine_cover >= 1.0
        or terrain["elevation_p90_m"] >= 3_000
        or (terrain["elevation_p90_m"] >= 2_500 and terrain["slope_p90_deg"] >= 15)
    ):
        return "alpine"
    if terrain["elevation_p90_m"] >= 1_500 or terrain["slope_p90_deg"] >= 15:
        return "mountain"
    if cover.get("built_up", 0.0) >= 12.0:
        return "urban"
    if cover.get("bare_sparse", 0.0) >= 35.0:
        return "desert_bare"
    if cover.get("tree", 0.0) >= 40.0:
        return "forest"
    if cover.get("cropland", 0.0) >= 40.0:
        return "cropland"
    water_cover = (
        cover.get("water", 0.0)
        + cover.get("wetland", 0.0)
        + cover.get("mangrove", 0.0)
    )
    if water_cover >= 25.0:
        return "water_wetland"
    if cover.get("shrubland", 0.0) + cover.get("grassland", 0.0) >= 40.0:
        return "shrub_grass"
    return "mixed"


def classify_esri_landscape(
    cover: Mapping[str, float], terrain: Mapping[str, float]
) -> str:
    """Assign a collection stratum from Esri 2025 annual land cover plus DEM."""

    if (
        cover.get("snow_ice", 0.0) >= 1.0
        or terrain["elevation_p90_m"] >= 3_000
        or (terrain["elevation_p90_m"] >= 2_500 and terrain["slope_p90_deg"] >= 15)
    ):
        return "alpine"
    if terrain["elevation_p90_m"] >= 1_500 or terrain["slope_p90_deg"] >= 15:
        return "mountain"
    if cover.get("built", 0.0) >= 12.0:
        return "urban"
    if cover.get("bare", 0.0) >= 35.0:
        return "desert_bare"
    if cover.get("trees", 0.0) >= 40.0:
        return "forest"
    if cover.get("crops", 0.0) >= 40.0:
        return "cropland"
    if cover.get("water", 0.0) + cover.get("flooded_vegetation", 0.0) >= 25.0:
        return "water_wetland"
    if cover.get("rangeland", 0.0) >= 40.0:
        return "shrub_grass"
    return "mixed"


# def normalize_administrative_name(value: object) -> str:
#     normalized = " ".join(str(value).strip().casefold().replace("&", "and").split())
#     aliases = {
#         "jammu & kashmir": "jammu and kashmir",
#         "jammu kashmir": "jammu and kashmir",
#         "uttaranchal": "uttarakhand",
#     }
#     return aliases.get(normalized, normalized)
import unicodedata


def normalize_administrative_name(value: object) -> str:
    text = str(value).strip()

    # Remove Unicode diacritics:
    # Himāchal -> Himachal
    # Kashmīr  -> Kashmir
    # Ladākh   -> Ladakh
    # Uttarākhand -> Uttarakhand
    text = unicodedata.normalize("NFKD", text)
    text = "".join(
        char for char in text
        if not unicodedata.combining(char)
    )

    normalized = " ".join(
        text.casefold()
            .replace("&", "and")
            .split()
    )

    aliases = {
        "jammu kashmir": "jammu and kashmir",
        "jammu and kashmir": "jammu and kashmir",
        "uttaranchal": "uttarakhand",
    }

    return aliases.get(normalized, normalized)

def priority_region(state_or_ut: object) -> str:
    state = normalize_administrative_name(state_or_ut)
    return (
        "western_himalaya"
        if state in WESTERN_HIMALAYA_STATES
        else "rest_of_india"
    )


def landscape_below_maximum(
    counts: Mapping[str, int],
    landscape: str,
    maximums: Mapping[str, int] = DEFAULT_LANDSCAPE_MAXIMUMS,
) -> bool:
    if landscape not in maximums:
        raise ValueError(f"Unknown landscape label: {landscape}")
    return int(counts.get(landscape, 0)) < int(maximums[landscape])


def landscape_slot_available(
    counts: Mapping[str, int],
    landscape: str,
    *,
    total: int,
    target_pairs: int,
    minimums: Mapping[str, int] = DEFAULT_LANDSCAPE_MINIMUMS,
) -> bool:
    """Reserve enough remaining slots to satisfy every outstanding class minimum."""

    return quota_slot_available(
        counts, landscape, total=total, target_pairs=target_pairs,
        minimums=minimums,
    )


def quota_slot_available(
    counts: Mapping[str, int],
    label: str,
    *,
    total: int,
    target_pairs: int,
    minimums: Mapping[str, int],
) -> bool:
    """Reserve remaining capacity for one mutually exclusive quota system."""

    if label not in minimums:
        raise ValueError(f"Unknown quota label: {label}")
    if total >= target_pairs:
        return False
    if int(counts.get(label, 0)) < int(minimums[label]):
        return True
    remaining_after_acceptance = target_pairs - total - 1
    outstanding_other = sum(
        max(int(required) - int(counts.get(name, 0)), 0)
        for name, required in minimums.items()
        if name != label
    )
    return remaining_after_acceptance >= outstanding_other


def combined_quota_slot_available(
    quota_systems: tuple[
        tuple[Mapping[str, int], str, Mapping[str, int]], ...
    ],
    *,
    total: int,
    target_pairs: int,
) -> bool:
    """Reserve capacity across overlapping landscape and geographic quotas."""

    if total >= target_pairs:
        return False
    for counts, label, minimums in quota_systems:
        if label not in minimums:
            raise ValueError(f"Unknown quota label: {label}")
    if any(
        int(counts.get(label, 0)) < int(minimums[label])
        for counts, label, minimums in quota_systems
    ):
        return True
    remaining_after_acceptance = target_pairs - total - 1
    outstanding_by_system = [
        sum(
            max(int(required) - int(counts.get(name, 0)), 0)
            for name, required in minimums.items()
            if name != label
        )
        for counts, label, minimums in quota_systems
    ]
    # One future pair can satisfy both a landscape and a geographic deficit, so
    # summing the systems would over-reserve capacity. The maximum is conservative
    # without assuming the deficits are disjoint.
    return remaining_after_acceptance >= max(outstanding_by_system, default=0)


def balanced_spatial_splits(
    frame: pd.DataFrame,
    *,
    block_column: str = "spatial_block_id",
    class_column: str = "landscape_class",
    seed: int = 42,
    fractions: Mapping[str, float] | None = None,
) -> dict[str, str]:
    """Assign complete spatial blocks while balancing class and total counts."""

    fractions = dict(fractions or {"train": 0.78, "val": 0.11, "test": 0.11})
    if set(fractions) != {"train", "val", "test"}:
        raise ValueError("fractions must contain train, val and test")
    if not np.isclose(sum(fractions.values()), 1.0):
        raise ValueError("split fractions must sum to one")
    if frame.empty:
        raise ValueError("cannot split an empty dataset")
    if frame[[block_column, class_column]].isna().any().any():
        raise ValueError("spatial block and landscape class values cannot be missing")
    blocks = sorted(frame[block_column].astype(str).unique())
    if len(blocks) < 3:
        raise ValueError("at least three spatial blocks are required")
    classes = sorted(frame[class_column].astype(str).unique())
    block_counts = pd.crosstab(frame[block_column].astype(str), frame[class_column].astype(str))
    block_counts = block_counts.reindex(index=blocks, columns=classes, fill_value=0)
    block_sizes = block_counts.sum(axis=1)
    total_by_class = block_counts.sum(axis=0).to_numpy(dtype=float)
    total_pairs = float(len(frame))
    split_names = ("test", "val", "train")
    target_total = {name: fractions[name] * total_pairs for name in split_names}
    target_class = {
        name: fractions[name] * total_by_class for name in split_names
    }
    current_total = {name: 0.0 for name in split_names}
    current_class = {name: np.zeros(len(classes), dtype=float) for name in split_names}
    assignments: dict[str, str] = {}

    rarity = np.divide(
        1.0,
        np.maximum(total_by_class, 1.0),
        out=np.zeros_like(total_by_class),
        where=total_by_class > 0,
    )
    order = sorted(
        blocks,
        key=lambda block: (
            -float(np.dot(block_counts.loc[block].to_numpy(dtype=float), rarity)),
            -int(block_sizes.loc[block]),
            hashlib.sha256(f"{seed}:{block}".encode()).hexdigest(),
        ),
    )
    # Seed validation and test with different blocks before greedy allocation.
    for split, block in zip(("test", "val", "train"), order[:3]):
        vector = block_counts.loc[block].to_numpy(dtype=float)
        assignments[block] = split
        current_total[split] += float(block_sizes.loc[block])
        current_class[split] += vector

    for block in order[3:]:
        vector = block_counts.loc[block].to_numpy(dtype=float)
        size = float(block_sizes.loc[block])
        scores = {}
        for split in split_names:
            class_need = np.maximum(target_class[split] - current_class[split], 0.0)
            class_gain = float(np.sum(np.minimum(vector, class_need) / np.maximum(target_class[split], 1.0)))
            size_need = max(target_total[split] - current_total[split], 0.0) / max(target_total[split], 1.0)
            overfill = max(current_total[split] + size - target_total[split], 0.0) / max(target_total[split], 1.0)
            tie = int(hashlib.sha256(f"{seed}:{block}:{split}".encode()).hexdigest()[:8], 16) / 2**32
            scores[split] = class_gain + 0.35 * size_need - 0.55 * overfill + tie * 1e-6
        selected = max(split_names, key=lambda name: scores[name])
        assignments[block] = selected
        current_total[selected] += size
        current_class[selected] += vector

    # Enforce the dataset-paper requirement that validation and test each contain
    # at least 10% of pairs. Complete blocks are moved; individual patches never
    # cross a spatial boundary to satisfy the count.
    minimum_holdout = int(np.ceil(total_pairs * 0.10))
    for holdout in ("test", "val"):
        while current_total[holdout] < minimum_holdout:
            train_blocks = [
                block for block, split in assignments.items() if split == "train"
            ]
            if not train_blocks:
                raise ValueError(f"cannot satisfy minimum size for {holdout}")
            class_need = np.maximum(
                target_class[holdout] - current_class[holdout], 0.0
            )
            selected = max(
                train_blocks,
                key=lambda block: (
                    float(np.sum(
                        np.minimum(
                            block_counts.loc[block].to_numpy(dtype=float), class_need
                        ) / np.maximum(target_class[holdout], 1.0)
                    )),
                    -int(block_sizes.loc[block]),
                    hashlib.sha256(
                        f"{seed}:holdout:{holdout}:{block}".encode()
                    ).hexdigest(),
                ),
            )
            vector = block_counts.loc[selected].to_numpy(dtype=float)
            size = float(block_sizes.loc[selected])
            assignments[selected] = holdout
            current_total["train"] -= size
            current_class["train"] -= vector
            current_total[holdout] += size
            current_class[holdout] += vector
    return assignments


def split_balance_table(
    frame: pd.DataFrame,
    *,
    class_column: str = "landscape_class",
    split_column: str = "split",
) -> pd.DataFrame:
    table = pd.crosstab(frame[class_column], frame[split_column])
    table = table.reindex(columns=["train", "val", "test"], fill_value=0)
    totals = table.sum(axis=1).replace(0, np.nan)
    for split in ("train", "val", "test"):
        table[f"{split}_percent_of_class"] = table[split] / totals * 100.0
    return table.reset_index()


def missing_landscape_reserves(
    counts: Mapping[str, int],
    minimums: Mapping[str, int] = DEFAULT_LANDSCAPE_MINIMUMS,
) -> dict[str, int]:
    return {
        name: max(int(required) - int(Counter(counts).get(name, 0)), 0)
        for name, required in minimums.items()
    }

In [ ]:
import os, csv, json, random, time, warnings, shutil, zipfile
import datetime as dt
from collections import Counter
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
import numpy as np, pandas as pd
import requests, shapely, geopandas as gpd
from shapely.geometry import shape
import rasterio
from rasterio.windows import Window
from rasterio.enums import Resampling
from rasterio.vrt import WarpedVRT
from rasterio.transform import Affine
from pyproj import Transformer
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from IPython.display import FileLink, Image, display
import planetary_computer as pc
from pystac_client import Client
warnings.filterwarnings('ignore')

# ---- Kaggle output / restart ----
OUT_DIR       = Path('/kaggle/working/SR_pairs_india_ESRI2025_4800_balanced')
# Restore only an output made by this exact v4 protocol.
# Example: Path('/kaggle/input/my-v4-output/SR_pairs_india_ESRI2025_4800_balanced')
RESTORE_FROM  = None
TARGET_PAIRS  = 4800
CREATE_ARCHIVE = False                                # ZIP duplicates the TIFF disk usage

# ---- Time / matching ----
START_DATE, END_DATE = '2025-01-01', '2026-09-15'
WINDOW_DAYS      = 8            # Landsat 8+9 combined revisit
MAX_DAY_GAP      = 0            # 0 = same UTC day only; 1-2 = allow near same-day
MAX_TIME_DIFF_MIN = 180         # (same-day mode) max minutes between L and S2 acquisition

# ---- Scene-level filters ----
MAX_CLOUD_L, MAX_CLOUD_S = 60, 60       # relaxed scene filter; patch QC remains >=99% clear
MIN_SUN_ELEV = 20                       # retain more high-latitude acquisitions

# ---- Patch geometry ----
LR_SIZE, FACTOR = 160, 3
LR_RES, HR_RES  = 30, 10
HR_SIZE = LR_SIZE * FACTOR

# ---- Patch-level QC ----
MIN_CLEAR        = 0.99         # joint clear fraction (Landsat AND Sentinel-2)
MIN_CLEAR_COARSE = 0.95         # cheap pre-test on decimated Landsat QA
S2_CLEAR_SCL     = [4, 5, 6, 11] # vegetation, bare, water, snow/ice
MAX_WATER_FRAC   = 0.50         # skip mostly-water patches
MIN_NIR_STD      = 150          # normal terrain
MIN_NIR_STD_ALPINE = 40         # snow/ice is naturally lower-texture
MIN_CORR         = 0.80         # normal terrain
MIN_CORR_ALPINE  = 0.65         # still rejects gross alpine misregistration
# CAND_PER_PAIR    = 40           # candidate cells tested per (Landsat, S2) pair
CAND_PER_PAIR    = 80           # candidate cells tested per (Landsat, S2) pair
MAX_PER_PAIR     = 12           # patches kept per pair (spatial diversity)

# ---- "Strictly inside India" ----
INDIA_BOUNDARY_FILE   = None    # optional: path/URL of your own India boundary (e.g. official SoI)
BORDER_MARGIN_DEG     = 0.03    # ~3.3 km: boundary is shrunk by this before testing
FOOTPRINT_MARGIN_DEG  = 0.004   # ~450 m: shrink scene footprints (avoids nodata edges)

# ---- Runtime / display ----
N_WORKERS       = 8
SHOW_EVERY      = 50            # show a live LR/HR preview every N accepted patches (0 = off)
DASHBOARD_EVERY = 400           # redraw the dashboard every N accepted patches (0 = off)
SEED            = 42

NAMES = ['blue', 'green', 'red', 'nir']
L_ASSETS = ['blue', 'green', 'red', 'nir08']
S_ASSETS = ['B02', 'B03', 'B04', 'B08']
INDIA_BBOX = [68.1, 6.7, 97.5, 37.1]
CS = LR_SIZE // 8                         # px per cell in the 1/8-decimated QA read
STAC_URL = 'https://planetarycomputer.microsoft.com/api/stac/v1'

if not os.environ.get('KAGGLE_KERNEL_RUN_TYPE'):
    print('WARNING: this notebook is configured for Kaggle paths.')
if RESTORE_FROM is not None:
    RESTORE_FROM = Path(RESTORE_FROM)
    if not RESTORE_FROM.is_dir():
        raise FileNotFoundError(f'RESTORE_FROM does not exist: {RESTORE_FROM}')
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    shutil.copytree(RESTORE_FROM, OUT_DIR, dirs_exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)
os.makedirs(f'{OUT_DIR}/landsat', exist_ok=True)
os.makedirs(f'{OUT_DIR}/sentinel', exist_ok=True)
MANIFEST = OUT_DIR / 'manifest.csv'
WINDOWS_DONE = OUT_DIR / 'windows_done.txt'
REJECTION_COUNTS = OUT_DIR / 'rejection_counts.json'
LANDSCAPE_COUNTS = OUT_DIR / 'landscape_counts.json'
GEOGRAPHIC_COUNTS = OUT_DIR / 'geographic_counts.json'
PRIORITY_COUNTS = OUT_DIR / 'priority_region_counts.json'
NORTH_WINDOWS_DONE = OUT_DIR / 'western_himalaya_windows_done.txt'
INDIA_WINDOWS_DONE = OUT_DIR / 'india_windows_done.txt'
PROTOCOL_PATH = OUT_DIR / 'protocol.json'
random.seed(SEED); np.random.seed(SEED)

T0 = time.time()
def log(msg):
    tqdm.write(f'[{time.strftime("%H:%M:%S")} +{(time.time()-T0)/60:5.1f} min] {msg}')

GDAL_OPTS = dict(GDAL_DISABLE_READDIR_ON_OPEN='EMPTY_DIR', VSI_CACHE='TRUE',
                 GDAL_HTTP_MAX_RETRY='4', GDAL_HTTP_RETRY_DELAY='1',
                 GDAL_HTTP_MULTIPLEX='YES', GDAL_HTTP_MERGE_CONSECUTIVE_RANGES='YES')
def rio_env():
    return rasterio.Env(**GDAL_OPTS)

def retry(fn, tries=4, base=1.7):
    for a in range(tries):
        try:
            return fn()
        except Exception:
            if a == tries - 1:
                raise
            time.sleep(base ** a)

if not os.environ.get('KAGGLE_KERNEL_RUN_TYPE'):
    print('Kaggle detection variable is absent; paths still point to /kaggle/working.')
print('Output:', OUT_DIR)
print('Internet must be enabled for STAC searches and signed COG reads.')
print('Free working space (GB):', round(shutil.disk_usage('/kaggle/working').free / 2**30, 2))

ESRI_LANDCOVER_SERVICE = (
    'https://ic.imagery1.arcgis.com/arcgis/rest/services/'
    'Sentinel2_10m_LandCover/ImageServer'
)
ESRI_LANDCOVER_YEAR = 2025
MAX_ESRI_CLOUD_PCT = 5.0

PROTOCOL = {
    'protocol_id': 'pc_india_rgbnir_esri2025_dem_himalaya_first_v4',
    'target_pairs': TARGET_PAIRS,
    'acquisition_interval': [START_DATE, END_DATE],
    'landcover_service': ESRI_LANDCOVER_SERVICE,
    'landcover_year': ESRI_LANDCOVER_YEAR,
    'worldcover_collection': 'esa-worldcover',
    'worldcover_year': 2021,
    'dem_collection': 'cop-dem-glo-30',
    'landscape_minimums': DEFAULT_LANDSCAPE_MINIMUMS,
    'landscape_maximums': DEFAULT_LANDSCAPE_MAXIMUMS,
    'geographic_minimums': DEFAULT_GEOGRAPHIC_MINIMUMS,
    'priority_region_minimums': DEFAULT_PRIORITY_REGION_MINIMUMS,
    'collection_order': ['western_himalaya', 'balanced_india'],
    'snow_ice_is_clear': True,
    'split_fractions': {'train': 0.8, 'val': 0.1, 'test': 0.1},
}
if PROTOCOL_PATH.is_file():
    previous_protocol = json.loads(PROTOCOL_PATH.read_text(encoding='utf-8'))
    if previous_protocol != PROTOCOL:
        raise ValueError(
            f'Existing output uses a different protocol: {PROTOCOL_PATH}. '
            'Use the v4 OUT_DIR. Older rows cannot be mixed because their acquisition '
            'years and land-cover labels differ.'
        )
else:
    PROTOCOL_PATH.write_text(json.dumps(PROTOCOL, indent=2), encoding='utf-8')


In [ ]:
def load_geoboundaries(level):
    js = requests.get(f'https://www.geoboundaries.org/api/current/gbOpen/IND/{level}/', timeout=60).json()
    js = js[0] if isinstance(js, list) else js
    return gpd.read_file(js.get('simplifiedGeometryGeoJSON') or js['gjDownloadURL']).to_crs(4326)

def load_india():
    if INDIA_BOUNDARY_FILE:
        return gpd.read_file(INDIA_BOUNDARY_FILE).to_crs(4326), 'user file'
    try:
        return load_geoboundaries('ADM0'), 'geoBoundaries gbOpen (IND ADM0)'
    except Exception as e:
        log(f'geoBoundaries failed ({str(e)[:60]}); falling back to Natural Earth')
        z = requests.get('https://naciscdn.org/naturalearth/10m/cultural/ne_10m_admin_0_countries.zip', timeout=120)
        open('/tmp/ne.zip', 'wb').write(z.content)
        g = gpd.read_file('/tmp/ne.zip')
        return g[g['ADMIN'] == 'India'].to_crs(4326), 'Natural Earth 10m'

india_gdf, src_name = load_india()
india = shapely.make_valid(shapely.union_all(india_gdf.geometry.values))
india_shr = shapely.make_valid(india.buffer(-BORDER_MARGIN_DEG))
shapely.prepare(india); shapely.prepare(india_shr)
area = gpd.GeoSeries([india], crs=4326).to_crs(6933).area.iloc[0] / 1e6
log(f'India boundary: {src_name} | area {area:,.0f} km2 | '
    f'shrunk by {BORDER_MARGIN_DEG} deg (~{BORDER_MARGIN_DEG*111:.1f} km) so patches stay strictly inside')

try:
    adm1 = load_geoboundaries('ADM1')            # states, only used for the dashboard
except Exception:
    adm1 = None

In [ ]:

catalog = Client.open(STAC_URL)

if adm1 is None or adm1.empty:
    raise RuntimeError(
        'ADM1 boundaries are required to enforce the 800-pair western-Himalaya reserve.'
    )
ADM1_NAME_COLUMN = next(
    (name for name in ('shapeName', 'NAME_1', 'name', 'NAME') if name in adm1.columns),
    None,
)
if ADM1_NAME_COLUMN is None:
    raise RuntimeError(f'No state-name field found in ADM1 columns: {list(adm1.columns)}')
ADMIN_LOOKUP = [
    (str(row[ADM1_NAME_COLUMN]), shapely.make_valid(row.geometry))
    for _, row in adm1.to_crs(4326).iterrows()
    if row.geometry is not None and not row.geometry.is_empty
]
WESTERN_HIMALAYA_GEOMETRY = shapely.make_valid(shapely.union_all([
    geometry for name, geometry in ADMIN_LOOKUP
    if normalize_administrative_name(name) in WESTERN_HIMALAYA_STATES
]))
if WESTERN_HIMALAYA_GEOMETRY.is_empty:
    raise RuntimeError(
        'ADM1 data did not contain Jammu and Kashmir, Ladakh, Himachal Pradesh, '
        'or Uttarakhand.'
    )

def state_or_ut_at(longitude, latitude):
    point = shapely.Point(float(longitude), float(latitude))
    matches = [name for name, geometry in ADMIN_LOOKUP if geometry.covers(point)]
    return matches[0] if matches else 'unassigned'

def source_pair_is_western_himalaya(landsat, sentinel):
    overlap = shapely.make_valid(
        shape(landsat.geometry).intersection(shape(sentinel.geometry))
    )
    return overlap.intersects(WESTERN_HIMALAYA_GEOMETRY)

service_metadata = requests.get(
    ESRI_LANDCOVER_SERVICE, params={'f': 'json'}, timeout=60
)
service_metadata.raise_for_status()
service_metadata = service_metadata.json()
if 'error' in service_metadata:
    raise RuntimeError(f'Esri land-cover service error: {service_metadata["error"]}')
year_fields = {field.get('name') for field in service_metadata.get('fields', [])}
if 'Year' not in year_fields:
    raise RuntimeError('Esri annual land-cover ImageServer has no Year field')


def stac_items(collection, date_range, cloud_max, bbox=None, intersects=None):
    def go():
        s = catalog.search(collections=[collection], datetime=date_range, bbox=bbox,
                           intersects=intersects, limit=200,
                           query={'eo:cloud_cover': {'lt': cloud_max}})
        return list(s.items())
    return retry(go)

def landsat_in_india(d0, d1):
    out = []
    for it in stac_items('landsat-c2-l2', f'{d0}/{d1}', MAX_CLOUD_L, bbox=INDIA_BBOX):
        p = it.properties
        if p.get('platform') not in ('landsat-8', 'landsat-9'): continue
        if p.get('landsat:collection_category') != 'T1': continue
        if p.get('view:sun_elevation', 90) < MIN_SUN_ELEV: continue
        if not shape(it.geometry).intersects(india_shr): continue
        out.append(it)
    return out

def sentinel_same_day(l_item):
    d = l_item.datetime.date()
    rng = f'{d - dt.timedelta(days=MAX_DAY_GAP)}/{d + dt.timedelta(days=MAX_DAY_GAP)}'
    try:
        items = stac_items('sentinel-2-l2a', rng, MAX_CLOUD_S, intersects=l_item.geometry)
    except Exception:
        return []
    if MAX_DAY_GAP == 0:
        items = [s for s in items
                 if abs((s.datetime - l_item.datetime).total_seconds()) <= MAX_TIME_DIFF_MIN * 60]
    return items

def preflight():
    log('Preflight: STAC search, URL signing and a COG read ...')
    its = landsat_in_india('2025-01-01', '2025-01-09')
    assert its, 'STAC returned no Landsat scenes; check internet / Planetary Computer status'
    href = pc.sign(its[0].assets['red'].href)
    with rio_env(), rasterio.open(href) as src:
        src.read(1, window=Window(0, 0, 8, 8))
    log(f'Preflight OK: {len(its)} Landsat scenes over India in the first test window')
preflight()

ANCILLARY_CACHE = {}

def ancillary_items(s_item):
    '''Find historical WorldCover 2021 and GLO-30 comparison tiles.'''
    cache_key = s_item.properties.get('s2:mgrs_tile', s_item.id)
    if cache_key in ANCILLARY_CACHE:
        return ANCILLARY_CACHE[cache_key]

    def search(collection, asset, date_range=None):
        query = catalog.search(
            collections=[collection], intersects=s_item.geometry,
            datetime=date_range, limit=100,
        )
        items = list(query.items())
        if collection == 'esa-worldcover':
            preferred = [item for item in items if '2021' in item.id]
            items = preferred or items
        hrefs = [pc.sign(item.assets[asset].href) for item in items if asset in item.assets]
        if not hrefs:
            raise RuntimeError(f'No {collection}/{asset} assets intersect {cache_key}')
        return hrefs

    result = {
        'worldcover': retry(lambda: search(
            'esa-worldcover', 'map', '2021-01-01/2021-12-31'
        )),
        'dem': retry(lambda: search('cop-dem-glo-30', 'data')),
    }
    ANCILLARY_CACHE[cache_key] = result
    return result


In [ ]:
def prepare_landsat(item):
    """Reads the grid + a 1/8-decimated QA mask once per Landsat scene, and pre-computes
    which 160-px cells lie fully inside India and the Landsat footprint."""
    hrefs = {k: pc.sign(item.assets[k].href) for k in L_ASSETS + ['qa_pixel']}
    with rio_env(), rasterio.open(hrefs['qa_pixel']) as src:
        tf, crs, H, W = src.transform, src.crs, src.height, src.width
        nr, nc = H // LR_SIZE, W // LR_SIZE
        qa = src.read(1, window=Window(0, 0, nc * LR_SIZE, nr * LR_SIZE),
                      out_shape=(nr * CS, nc * CS), resampling=Resampling.nearest)
    clear = ((qa & 31) == 0).reshape(nr, CS, nc, CS).mean((1, 3))   # fill|dil.cloud|cirrus|cloud|shadow

    # 3x3 sample points per cell (corners, edge mid-points, centre), in lon/lat
    offs = np.array([0, LR_SIZE // 2, LR_SIZE])
    jj, ii = np.meshgrid(np.arange(nr), np.arange(nc), indexing='ij')
    cols = np.broadcast_to(ii[..., None, None] * LR_SIZE + offs[None, None, None, :], (nr, nc, 3, 3))
    rows = np.broadcast_to(jj[..., None, None] * LR_SIZE + offs[None, None, :, None], (nr, nc, 3, 3))
    xs = tf.c + tf.a * cols + tf.b * rows
    ys = tf.f + tf.d * cols + tf.e * rows
    lon, lat = Transformer.from_crs(crs, 4326, always_xy=True).transform(xs.ravel(), ys.ravel())

    region = shapely.intersection(india_shr, shape(item.geometry).buffer(-FOOTPRINT_MARGIN_DEG))
    if region.is_empty:
        base_ok = np.zeros((nr, nc), bool)
    else:
        shapely.prepare(region)
        base_ok = shapely.contains_xy(region, lon, lat).reshape(nr, nc, 9).all(-1)
    return dict(item=item, name=item.id, hrefs=hrefs, tf=tf, crs=crs, nr=nr, nc=nc,
                clear=clear, lon=np.asarray(lon), lat=np.asarray(lat), base_ok=base_ok)

def prepare_sentinel(s_item):
    base = float(s_item.properties.get('s2:processing_baseline', '04.00') or 4.0)
    return dict(item=s_item, hrefs={k: pc.sign(s_item.assets[k].href) for k in S_ASSETS + ['SCL']},
                offset=1000 if base >= 4.0 else 0)          # harmonise baseline >= 04.00

# def select_cells(sc, sd):
#     Sg = shape(sd['item'].geometry).buffer(-FOOTPRINT_MARGIN_DEG)
#     if Sg.is_empty:
#         return [], 0
#     shapely.prepare(Sg)
#     s_ok = shapely.contains_xy(Sg, sc['lon'], sc['lat']).reshape(sc['nr'], sc['nc'], 9).all(-1)
#     geo = sc['base_ok'] & s_ok
#     ok = geo & (sc['clear'] >= MIN_CLEAR_COARSE)
#     cells = [(int(i), int(j)) for j, i in zip(*np.nonzero(ok)) if (sc['name'], int(i), int(j)) not in seen]
#     random.shuffle(cells)
#     return cells[:CAND_PER_PAIR], int(geo.sum())

def select_cells(sc, sd):
    """
    Select only spatially NEW candidate cells.

    Important:
    - removes already-saved geographic footprints BEFORE build_patch()
    - preserves the existing >=50% overlap rejection rule
    - existing TIFFs/manifest are untouched
    """

    Sg = shape(sd['item'].geometry).buffer(-FOOTPRINT_MARGIN_DEG)

    if Sg.is_empty:
        return [], 0, 0

    shapely.prepare(Sg)

    nr, nc = sc['nr'], sc['nc']

    # Sentinel footprint validity
    s_ok = (
        shapely.contains_xy(
            Sg,
            sc['lon'],
            sc['lat']
        )
        .reshape(nr, nc, 9)
        .all(-1)
    )

    geo = sc['base_ok'] & s_ok

    # Cheap Landsat QA filter
    ok = geo & (sc['clear'] >= MIN_CLEAR_COARSE)

    # We already calculated 3x3 lon/lat samples for every Landsat cell.
    # index 4 = centre of the 3x3 samples.
    center_lon = sc['lon'].reshape(nr, nc, 9)[:, :, 4]
    center_lat = sc['lat'].reshape(nr, nc, 9)[:, :, 4]

    fresh_cells = []
    already_covered = 0

    rows, cols = np.nonzero(ok)

    for j, i in zip(rows, cols):

        i = int(i)
        j = int(j)

        # Same Landsat product + exact cell already used
        if (sc['name'], i, j) in seen:
            already_covered += 1
            continue

        lon = float(center_lon[j, i])
        lat = float(center_lat[j, i])

        # Same equal-area coordinate system already used by manifest
        equal_x, equal_y = equal_area_xy(lon, lat)

        # IMPORTANT:
        # reject geographically represented cells NOW,
        # before downloading S2 bands / Esri / DEM / WorldCover.
        if overlaps_saved_location(equal_x, equal_y):
            already_covered += 1
            continue

        fresh_cells.append((i, j))

    # Randomness among genuinely unused coordinates only
    random.shuffle(fresh_cells)

    return (
        fresh_cells[:CAND_PER_PAIR],
        int(geo.sum()),
        already_covered,
    )

In [ ]:
def read_l(href, win):
    with rasterio.open(href) as src:
        return src.read(1, window=win)

def read_s2(href, crs, hr_tf, resampling):
    with rasterio.open(href) as src:
        with WarpedVRT(src, crs=crs, transform=hr_tf, width=HR_SIZE, height=HR_SIZE,
                       resampling=resampling, src_nodata=0, nodata=0) as v:
            return v.read(1)


def read_mosaic(hrefs, crs, transform, width, height, resampling):
    '''Read every intersecting ancillary COG into one exact patch grid.'''
    output = np.full((height, width), np.nan, dtype=np.float32)
    for href in hrefs:
        try:
            with rasterio.open(href) as src:
                options = dict(
                    crs=crs, transform=transform, width=width, height=height,
                    resampling=resampling, nodata=np.nan, dtype='float32',
                )
                if src.nodata is not None:
                    options['src_nodata'] = src.nodata
                with WarpedVRT(src, **options) as vrt:
                    values = vrt.read(1, masked=True).filled(np.nan)
            valid = np.isfinite(values)
            output[valid] = values[valid]
        except Exception:
            continue
    return output

def read_esri_landcover_2025(crs, transform, width, height):
    '''Read categorical 2025 land cover directly from the public ImageServer.'''
    epsg = crs.to_epsg()
    if epsg is None:
        raise ValueError(f'Land-cover request needs an EPSG CRS, found {crs}')
    left = float(transform.c)
    top = float(transform.f)
    right = left + float(transform.a) * width
    bottom = top + float(transform.e) * height
    mosaic_rule = {
        'mosaicMethod': 'esriMosaicAttribute',
        'where': f'Year = {ESRI_LANDCOVER_YEAR}',
        'sortField': 'Year',
        'sortValue': str(ESRI_LANDCOVER_YEAR),
        'ascending': False,
        'mosaicOperation': 'MT_FIRST',
    }
    response = requests.get(
        f'{ESRI_LANDCOVER_SERVICE}/exportImage',
        params={
            'bbox': f'{left},{bottom},{right},{top}',
            'bboxSR': epsg,
            'imageSR': epsg,
            'size': f'{width},{height}',
            'format': 'tiff',
            'pixelType': 'U8',
            'interpolation': 'RSP_NearestNeighbor',
            'mosaicRule': json.dumps(mosaic_rule, separators=(',', ':')),
            'f': 'json',
        },
        timeout=120,
    )
    response.raise_for_status()
    payload = response.json()
    if 'error' in payload:
        raise RuntimeError(f'Esri exportImage error: {payload["error"]}')
    href = payload.get('href')
    if not href:
        raise RuntimeError(f'Esri exportImage returned no href: {payload}')
    image_response = requests.get(href, timeout=120)
    image_response.raise_for_status()
    with rasterio.io.MemoryFile(image_response.content) as memory:
        with memory.open() as source:
            labels = source.read(
                1, out_shape=(height, width), resampling=Resampling.nearest
            )
    labels = np.asarray(labels, dtype=np.uint8)
    unexpected = set(np.unique(labels)) - {0, *ESRI_LANDCOVER_CLASSES}
    if unexpected:
        raise ValueError(f'Unexpected Esri land-cover codes: {sorted(unexpected)}')
    return labels

def equal_area_xy(longitude, latitude):
    radius = 6_371_007.181
    cosine_standard_parallel = np.cos(np.radians(30.0))
    x = radius * np.radians(longitude) * cosine_standard_parallel
    y = radius * np.sin(np.radians(latitude)) / cosine_standard_parallel
    return float(x), float(y)

def corr(a, b):

    a, b = a.ravel().astype('float64'), b.ravel().astype('float64')
    if a.std() < 1e-6 or b.std() < 1e-6: return np.nan
    return float(np.corrcoef(a, b)[0, 1])

def write_tif(path, arr, crs, tf, tags):
    with rasterio.open(path, 'w', driver='GTiff', height=arr.shape[1], width=arr.shape[2],
                       count=arr.shape[0], dtype='int16', crs=crs, transform=tf,
                       compress='deflate', predictor=2) as dst:
        dst.write(arr)
        dst.descriptions = tuple(NAMES)
        dst.update_tags(**tags)


def build_patch(sc, sd, ancillary, i, j, required_priority_region=None):
    try:
        with rio_env():
            col0, row0 = i * LR_SIZE, j * LR_SIZE
            tf, crs = sc['tf'], sc['crs']
            x_ul = tf.c + tf.a * col0 + tf.b * row0
            y_ul = tf.f + tf.d * col0 + tf.e * row0
            lr_tf = Affine(LR_RES, 0, x_ul, 0, -LR_RES, y_ul)
            hr_tf = Affine(HR_RES, 0, x_ul, 0, -HR_RES, y_ul)
            win = Window(col0, row0, LR_SIZE, LR_SIZE)
            cx = x_ul + LR_SIZE * LR_RES / 2
            cy = y_ul - LR_SIZE * LR_RES / 2
            clon, clat = Transformer.from_crs(
                crs, 4326, always_xy=True
            ).transform(cx, cy)
            state_or_ut = state_or_ut_at(clon, clat)
            region = priority_region(state_or_ut)
            if required_priority_region and region != required_priority_region:
                return dict(ok=False, why=f'outside_{required_priority_region}')

            # Cheap tests first: only QA + SCL are read.
            qa = retry(lambda: read_l(sc['hrefs']['qa_pixel'], win))
            l_clear = (qa & 31) == 0
            if l_clear.mean() < MIN_CLEAR:
                return dict(ok=False, why='cloud_landsat')
            scl = retry(lambda: read_s2(
                sd['hrefs']['SCL'], crs, hr_tf, Resampling.nearest
            ))
            s_clear = np.isin(scl, S2_CLEAR_SCL)
            joint = (
                np.repeat(np.repeat(l_clear, FACTOR, 0), FACTOR, 1) & s_clear
            ).mean()
            if joint < MIN_CLEAR:
                return dict(ok=False, why='cloud_joint')
            water = float((scl == 6).mean())
            if water > MAX_WATER_FRAC:
                return dict(ok=False, why='water')

            l_dn = np.stack([
                retry(lambda k=k: read_l(sc['hrefs'][k], win))
                for k in L_ASSETS
            ])
            s_dn = np.stack([
                retry(lambda k=k: read_s2(
                    sd['hrefs'][k], crs, hr_tf, Resampling.bilinear
                )) for k in S_ASSETS
            ])
            if (l_dn == 0).any(0).mean() > 0.001 or (s_dn == 0).any(0).mean() > 0.001:
                return dict(ok=False, why='nodata')
            lr = np.clip(
                l_dn.astype('float32') * 0.275 - 2000.0, 0, 10000
            ).round().astype('int16')
            hr = np.clip(
                s_dn.astype('float32') - sd['offset'], 0, 10000
            ).astype('int16')

            hr_ds = hr.astype('float32').reshape(
                4, LR_SIZE, FACTOR, LR_SIZE, FACTOR
            ).mean((2, 4))
            c_red, c_nir = corr(hr_ds[2], lr[2]), corr(hr_ds[3], lr[3])

            esri_labels = retry(lambda: read_esri_landcover_2025(
                crs, hr_tf, HR_SIZE, HR_SIZE,
            ))
            try:
                cover = esri_landcover_percentages(esri_labels)
            except ValueError:
                return dict(ok=False, why='esri2025_landcover_missing')
            if cover.get('clouds', 0.0) > MAX_ESRI_CLOUD_PCT:
                return dict(ok=False, why='esri2025_landcover_cloud')

            historical_labels = retry(lambda: read_mosaic(
                ancillary['worldcover'], crs, hr_tf, HR_SIZE, HR_SIZE,
                Resampling.nearest,
            ))
            historical_labels = np.where(
                np.isfinite(historical_labels), np.rint(historical_labels), 0
            ).astype(np.uint8)
            try:
                historical_cover = worldcover_percentages(historical_labels)
            except ValueError:
                return dict(ok=False, why='worldcover_missing')
            elevation = retry(lambda: read_mosaic(
                ancillary['dem'], crs, lr_tf, LR_SIZE, LR_SIZE,
                Resampling.bilinear,
            ))
            try:
                terrain = terrain_statistics(elevation, LR_RES)
            except ValueError:
                return dict(ok=False, why='dem_missing')
            landscape = classify_esri_landscape(cover, terrain)
            nir_threshold = (
                MIN_NIR_STD_ALPINE if landscape == 'alpine' else MIN_NIR_STD
            )
            correlation_threshold = (
                MIN_CORR_ALPINE if landscape == 'alpine' else MIN_CORR
            )
            if hr[3].std() < nir_threshold:
                return dict(ok=False, why=f'flat_{landscape}')
            if not (
                c_red >= correlation_threshold and c_nir >= correlation_threshold
            ):
                return dict(ok=False, why=f'low_corr_{landscape}')

            l_it, s_it = sc['item'], sd['item']
            tile = s_it.properties.get('s2:mgrs_tile', 'NA')
            pid = f"{l_it.id}__{tile}_{s_it.datetime:%Y%m%d}__{i}_{j}"
            equal_x, equal_y = equal_area_xy(clon, clat)
            geography = geographic_stratum(clat)
            row = dict(
                pair_id=pid, landsat_id=l_it.id, sentinel_id=s_it.id,
                platform=l_it.properties['platform'],
                s2_platform=s_it.properties.get('platform'), s2_tile=tile,
                date_L=f'{l_it.datetime:%Y-%m-%d}',
                date_S=f'{s_it.datetime:%Y-%m-%d}',
                day_gap=abs((l_it.datetime.date() - s_it.datetime.date()).days),
                time_diff_min=round(abs(
                    (s_it.datetime - l_it.datetime).total_seconds()
                ) / 60, 1),
                cloud_L=l_it.properties.get('eo:cloud_cover'),
                cloud_S=s_it.properties.get('eo:cloud_cover'),
                crs=str(crs), ul_x=x_ul, ul_y=y_ul,
                center_lon=round(clon, 6), center_lat=round(clat, 6),
                equal_area_x_m=round(equal_x, 2),
                equal_area_y_m=round(equal_y, 2),
                clear_frac=round(float(joint), 4),
                water_frac=round(water, 4), corr_red=round(c_red, 4),
                corr_nir=round(c_nir, 4), nir_std=round(float(hr[3].std()), 1),
                landscape_class=landscape,
                geographic_stratum=geography,
                state_or_ut=state_or_ut,
                priority_region=region,
                landcover_year=ESRI_LANDCOVER_YEAR,
                landcover_source='Esri Impact Observatory Sentinel-2 10m Annual LULC',
                landcover_service=ESRI_LANDCOVER_SERVICE,
                worldcover_year=2021,
                worldcover_source='ESA WorldCover 2021 v200 historical comparison',
                dem_source='Copernicus DEM GLO-30',
                **{f'esri2025_{name}_pct': round(value, 4)
                   for name, value in cover.items()},
                **{f'wc2021_{name}_pct': round(value, 4)
                   for name, value in historical_cover.items()},
                **{name: round(value, 4) for name, value in terrain.items()},
            )
            tags = dict(
                landsat=l_it.id, sentinel=s_it.id,
                landscape=landscape, landcover='Esri_2025',
                worldcover_comparison='2021_v200', state_or_ut=state_or_ut,
            )
            return dict(
                ok=True, row=row, preview=(lr, hr), cell=(i, j),
                low=lr, high=hr, crs=crs, low_transform=lr_tf,
                high_transform=hr_tf, tags=tags,
            )
    except Exception as error:
        return dict(ok=False, why='read_error', err=str(error)[:160])


In [ ]:
def rgb(a, lo, hi, idx=(2, 1, 0)):
    x = a[list(idx)].astype('float32').transpose(1, 2, 0)
    return np.clip((x - lo) / (hi - lo + 1e-6), 0, 1)

def show_pair(lr, hr, title=''):
    lo, hi = np.percentile(hr[[2, 1, 0]], 2), np.percentile(hr[[2, 1, 0]], 98)
    lo2, hi2 = np.percentile(hr[[3, 2, 1]], 2), np.percentile(hr[[3, 2, 1]], 98)
    fig, ax = plt.subplots(1, 3, figsize=(14, 4.6))
    ax[0].imshow(rgb(lr, lo, hi), interpolation='nearest'); ax[0].set_title('Landsat 30 m, 160x160 (input)')
    ax[1].imshow(rgb(hr, lo, hi)); ax[1].set_title('Sentinel-2 10 m, 480x480 (target)')
    ax[2].imshow(rgb(hr, lo2, hi2, (3, 2, 1))); ax[2].set_title('Sentinel-2 false colour (NIR-R-G)')
    for a in ax: a.axis('off')
    fig.suptitle(title, fontsize=10); plt.tight_layout(); plt.show()

def dashboard():
    if not os.path.exists(MANIFEST): return
    df = pd.read_csv(MANIFEST)
    if df.empty: return
    df['dt'] = pd.to_datetime(df['date_L'])
    fig = plt.figure(figsize=(17, 13)); gs = fig.add_gridspec(3, 3)

    ax = fig.add_subplot(gs[0:2, 0:2])
    if adm1 is not None: adm1.boundary.plot(ax=ax, color='0.8', lw=0.3)
    gpd.GeoSeries([india], crs=4326).boundary.plot(ax=ax, color='k', lw=0.9)
    sc_ = ax.scatter(df.center_lon, df.center_lat, c=df.dt.dt.month, cmap='twilight', s=9, vmin=1, vmax=12)
    plt.colorbar(sc_, ax=ax, shrink=0.6, label='acquisition month')
    ax.set_title(f'{len(df)} patches  |  L8: {(df.platform=="landsat-8").sum()}  L9: {(df.platform=="landsat-9").sum()}')
    ax.set_aspect('equal')

    ax = fig.add_subplot(gs[0, 2]); m = df.dt.dt.month.value_counts().reindex(range(1, 13), fill_value=0)
    ax.bar(m.index, m.values, color='tab:blue'); ax.set_title('Patches per month of year'); ax.set_xticks(range(1, 13))

    ax = fig.add_subplot(gs[1, 2])
    ax.hist(df.corr_red, 30, alpha=.6, label='red'); ax.hist(df.corr_nir, 30, alpha=.6, label='NIR')
    ax.set_title('LR vs block-mean(HR) correlation'); ax.legend()

    ax = fig.add_subplot(gs[2, 0]); t = df.groupby(df.dt.dt.to_period('M')).size()
    ax.bar([str(p) for p in t.index], t.values, color='tab:green')
    ax.set_title('Patches per year-month'); ax.tick_params(axis='x', rotation=90, labelsize=6)

    ax = fig.add_subplot(gs[2, 1]); ax.hist(df.time_diff_min, 30, color='tab:orange')
    ax.set_title(f'Landsat-Sentinel time gap (min) | day_gap>0: {(df.day_gap>0).sum()}')

    ax = fig.add_subplot(gs[2, 2])
    if adm1 is not None:
        pts = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df.center_lon, df.center_lat), crs=4326)
        j = gpd.sjoin(pts, adm1[['shapeName', 'geometry']], predicate='within', how='left')
        s = j['shapeName'].fillna('unknown').value_counts().head(12)[::-1]
        ax.barh(s.index, s.values, color='tab:purple'); ax.set_title('Top states/UTs')
    else:
        ax.axis('off')
    plt.tight_layout(); plt.show()

def gallery(n=4):
    files = sorted(os.listdir(f'{OUT_DIR}/landsat'))
    for f in random.sample(files, min(n, len(files))):
        with rasterio.open(f'{OUT_DIR}/landsat/{f}') as a, rasterio.open(f'{OUT_DIR}/sentinel/{f}') as b:
            show_pair(a.read(), b.read(), f[:-4])

def verify_inside_india():
    """Checks all 4 corners of every saved patch against the UNSHRUNK boundary."""
    df = pd.read_csv(MANIFEST); bad = 0; size = LR_SIZE * LR_RES
    for crs, g in df.groupby('crs'):
        tr = Transformer.from_crs(crs, 4326, always_xy=True)
        xs = np.stack([g.ul_x, g.ul_x + size, g.ul_x + size, g.ul_x], 1)
        ys = np.stack([g.ul_y, g.ul_y, g.ul_y - size, g.ul_y - size], 1)
        lon, lat = tr.transform(xs.ravel(), ys.ravel())
        bad += int((~shapely.contains_xy(india, lon, lat).reshape(-1, 4).all(1)).sum())
    log(f'Boundary check: {len(df) - bad}/{len(df)} patches fully inside India' + ('' if bad == 0 else f'  ({bad} outside!)'))

In [ ]:
START_DATE, END_DATE = '2026-04-15', '2026-05-31'

In [ ]:
FIELDS = [
    'pair_id', 'landsat_id', 'sentinel_id', 'platform', 's2_platform',
    's2_tile', 'date_L', 'date_S', 'day_gap', 'time_diff_min',
    'cloud_L', 'cloud_S', 'crs', 'ul_x', 'ul_y', 'center_lon',
    'center_lat', 'equal_area_x_m', 'equal_area_y_m', 'clear_frac',
    'water_frac', 'corr_red', 'corr_nir', 'nir_std',
    'landscape_class', 'geographic_stratum', 'state_or_ut', 'priority_region',
    'landcover_year', 'landcover_source', 'landcover_service',
    *ESRI_LANDCOVER_COLUMNS, 'worldcover_year', 'worldcover_source',
    'dem_source', *WORLDCOVER_COLUMNS, 'elevation_mean_m',
    'elevation_p90_m', 'elevation_max_m', 'slope_mean_deg',
    'slope_p90_deg',
]

# Online overlap control prevents repeated dates and overlapping source
# products from contributing the same 4.8 km footprint more than once.
SPATIAL_HASH_M = LR_SIZE * LR_RES
spatial_bins = {}

def spatial_bucket(x, y):
    x, y = float(x), float(y)
    return int(np.floor(x / SPATIAL_HASH_M)), int(np.floor(y / SPATIAL_HASH_M))

def register_center(x, y):
    x, y = float(x), float(y)
    if not (np.isfinite(x) and np.isfinite(y)):
        raise ValueError(f'Invalid equal-area center: {(x, y)}')
    spatial_bins.setdefault(spatial_bucket(x, y), []).append((x, y))

def overlaps_saved_location(x, y, threshold=0.50):
    bx, by = spatial_bucket(x, y)
    area = SPATIAL_HASH_M ** 2
    for dx in (-1, 0, 1):
        for dy in (-1, 0, 1):
            for old_x, old_y in spatial_bins.get((bx + dx, by + dy), []):
                overlap_x = max(SPATIAL_HASH_M - abs(float(x) - old_x), 0.0)
                overlap_y = max(SPATIAL_HASH_M - abs(float(y) - old_y), 0.0)
                if overlap_x * overlap_y / area >= threshold:
                    return True
    return False

seen, total = set(), 0
landscape_counts = Counter()
geographic_counts = Counter()
priority_counts = Counter()
if MANIFEST.is_file():
    existing_frame = pd.read_csv(MANIFEST)
    if 'geographic_stratum' not in existing_frame.columns:
        existing_frame['geographic_stratum'] = existing_frame.center_lat.map(
            geographic_stratum
        )
        temporary_manifest = MANIFEST.with_suffix('.north_recovery.tmp.csv')
        existing_frame.to_csv(temporary_manifest, index=False)
        temporary_manifest.replace(MANIFEST)
        print('Added geographic_stratum to the existing manifest; TIFFs were unchanged.')
    with MANIFEST.open(newline='', encoding='utf-8') as stream:
        reader = csv.DictReader(stream)
        missing_columns = sorted(set(FIELDS) - set(reader.fieldnames or []))
        if missing_columns:
            raise ValueError(
                f'Existing manifest predates the balanced protocol and lacks '
                f'{missing_columns}. Use the new OUT_DIR.'
            )
        for record in reader:
            i, j = record['pair_id'].split('__')[-1].split('_')
            seen.add((record['landsat_id'], int(i), int(j)))
            register_center(record['equal_area_x_m'], record['equal_area_y_m'])
            landscape_counts[record['landscape_class']] += 1
            geographic_counts[record['geographic_stratum']] += 1
            priority_counts[record['priority_region']] += 1
            total += 1
    log(f'Resuming: {total} unique patches already on disk')
north_done_windows = set(
    NORTH_WINDOWS_DONE.read_text(encoding='utf-8').split()
) if NORTH_WINDOWS_DONE.is_file() else set()
india_done_windows = set(
    INDIA_WINDOWS_DONE.read_text(encoding='utf-8').split()
) if INDIA_WINDOWS_DONE.is_file() else set()

new_file = not MANIFEST.exists()
manifest_stream = MANIFEST.open('a', newline='', encoding='utf-8')
writer = csv.DictWriter(manifest_stream, FIELDS)
if new_file:
    writer.writeheader()

windows, date = [], dt.date.fromisoformat(START_DATE)
final_date = dt.date.fromisoformat(END_DATE)
while date <= final_date:
    windows.append((
        date.isoformat(),
        min(date + dt.timedelta(days=WINDOW_DAYS - 1), final_date).isoformat(),
    ))
    date += dt.timedelta(days=WINDOW_DAYS)
random.shuffle(windows)
schedule = [
    ('western_himalaya', *window, NORTH_WINDOWS_DONE)
    for window in windows if window[0] not in north_done_windows
] + [
    ('balanced_india', *window, INDIA_WINDOWS_DONE)
    for window in windows if window[0] not in india_done_windows
]
log(
    f'{len(windows)} date windows, two passes | '
    f'target {TARGET_PAIRS} spatially unique patches | '
    f'mode: {"strict same-day" if MAX_DAY_GAP == 0 else f"max gap {MAX_DAY_GAP} d"}'
)
print('Landscape collection reserves:', DEFAULT_LANDSCAPE_MINIMUMS)
print('Landscape hard caps:', DEFAULT_LANDSCAPE_MAXIMUMS)
print('Geographic collection reserves:', DEFAULT_GEOGRAPHIC_MINIMUMS)
print('Priority region reserves:', DEFAULT_PRIORITY_REGION_MINIMUMS)

pool = ThreadPoolExecutor(N_WORKERS)
bar = tqdm(total=TARGET_PAIRS, initial=total, desc='balanced patches', unit='patch')
rej_total = Counter(
    json.loads(REJECTION_COUNTS.read_text(encoding='utf-8'))
) if REJECTION_COUNTS.is_file() else Counter()
last_dash = total
for window_index, (phase, window_start, window_end, phase_done_path) in enumerate(schedule, 1):
    if total >= TARGET_PAIRS:
        break
    if (
        phase == 'western_himalaya'
        and priority_counts['western_himalaya']
        >= DEFAULT_PRIORITY_REGION_MINIMUMS['western_himalaya']
    ):
        continue
    rejected, accepted_in_window = Counter(), 0
    try:
        log(
            f'[{phase} window {window_index}/{len(schedule)}] {window_start} -> '
            f'{window_end}: searching Landsat ...'
        )
        landsat_items = landsat_in_india(window_start, window_end)
        matches = list(pool.map(sentinel_same_day, landsat_items))
    except Exception as error:
        log(f'window failed ({str(error)[:120]}), skipping')
        continue

    pairs = [
        (landsat, sentinel)
        for landsat, sentinels in zip(landsat_items, matches)
        for sentinel in sentinels
    ]
    if phase == 'western_himalaya':
        pairs = [
            pair for pair in pairs
            if source_pair_is_western_himalaya(*pair)
        ]
    log(
        f'{sum(bool(items) for items in matches)} Landsat scenes have '
        f'same-day Sentinel-2 -> {len(pairs)} source-product pairs'
    )
    random.shuffle(landsat_items)
    order = {item.id: index for index, item in enumerate(landsat_items)}
    def collection_priority(pair):
        landsat, sentinel = pair
        max_latitude = max(
            shape(landsat.geometry).bounds[3],
            shape(sentinel.geometry).bounds[3],
        )
        return (
            0 if max_latitude >= 33.0 else 1,
            -max_latitude,
            order[landsat.id],
        )
    pairs.sort(key=collection_priority)

    scene_cache = {}
    for landsat_item, sentinel_item in pairs:
        if total >= TARGET_PAIRS:
            break
        try:
            if landsat_item.id not in scene_cache:
                scene_cache[landsat_item.id] = retry(
                    lambda: prepare_landsat(landsat_item)
                )
            # scene = scene_cache[landsat_item.id]
            # sentinel = prepare_sentinel(sentinel_item)
            # ancillary = ancillary_items(sentinel_item)
            # cells, geographic_cells = select_cells(scene, sentinel)
            scene = scene_cache[landsat_item.id]
            sentinel = prepare_sentinel(sentinel_item)
            
            cells, geographic_cells, already_covered = select_cells(
                scene,
                sentinel
            )
            
            rejected['already_covered_early'] += already_covered
            
            if geographic_cells == 0:
                rejected['no_cells_inside_india'] += 1
                continue
            
            if not cells:
                rejected['no_new_spatial_cells'] += 1
                continue
            
            # IMPORTANT:
            # Do ancillary STAC searches ONLY when this source pair
            # actually contains usable NEW geographic coordinates.
            ancillary = ancillary_items(sentinel_item)
        except Exception as error:
            rejected['prep_error'] += 1
            continue
        # if geographic_cells == 0:
        #     rejected['no_cells_inside_india'] += 1
        #     continue
        # if not cells:
        #     rejected['coarse_cloud'] += 1
        #     continue

        accepted_for_pair = 0
        for start in range(0, len(cells), 2 * N_WORKERS):
            if accepted_for_pair >= MAX_PER_PAIR or total >= TARGET_PAIRS:
                break
            batch = cells[start:start + 2 * N_WORKERS]
            results = pool.map(
                lambda cell: build_patch(
                    scene, sentinel, ancillary, *cell,
                    required_priority_region=(
                        'western_himalaya'
                        if phase == 'western_himalaya' else None
                    ),
                ),
                batch,
            )
            for result in results:
                if not result['ok']:
                    rejected[result['why']] += 1
                    continue
                row = result['row']
                if overlaps_saved_location(
                    row['equal_area_x_m'], row['equal_area_y_m']
                ):
                    rejected['overlapping_geographic_footprint'] += 1
                    continue
                landscape = row['landscape_class']
                geography = row['geographic_stratum']
                region = row['priority_region']
                if not landscape_below_maximum(
                    landscape_counts, landscape, DEFAULT_LANDSCAPE_MAXIMUMS
                ):
                    rejected[f'landscape_cap_{landscape}'] += 1
                    continue
                if not combined_quota_slot_available(
                    (
                        (landscape_counts, landscape, DEFAULT_LANDSCAPE_MINIMUMS),
                        (geographic_counts, geography, DEFAULT_GEOGRAPHIC_MINIMUMS),
                        (priority_counts, region, DEFAULT_PRIORITY_REGION_MINIMUMS),
                    ),
                    total=total, target_pairs=TARGET_PAIRS,
                ):
                    rejected[
                        f'quota_reserved_against_{landscape}_{geography}'
                    ] += 1
                    continue

                pair_id = row['pair_id']
                write_tif(
                    OUT_DIR / 'landsat' / f'{pair_id}.tif', result['low'],
                    result['crs'], result['low_transform'], result['tags'],
                )
                write_tif(
                    OUT_DIR / 'sentinel' / f'{pair_id}.tif', result['high'],
                    result['crs'], result['high_transform'], result['tags'],
                )
                writer.writerow(row)
                seen.add((scene['name'], *result['cell']))
                register_center(row['equal_area_x_m'], row['equal_area_y_m'])
                landscape_counts[landscape] += 1
                geographic_counts[geography] += 1
                priority_counts[region] += 1
                accepted_for_pair += 1
                total += 1
                accepted_in_window += 1
                bar.update(1)
                if SHOW_EVERY and (total == 1 or total % SHOW_EVERY == 0):
                    show_pair(*result['preview'], title=(
                        f'{pair_id} | {landscape} | '
                        f'{dict(landscape_counts)}'
                    ))
            manifest_stream.flush()
            LANDSCAPE_COUNTS.write_text(
                json.dumps(dict(landscape_counts), indent=2), encoding='utf-8'
            )
            GEOGRAPHIC_COUNTS.write_text(
                json.dumps(dict(geographic_counts), indent=2), encoding='utf-8'
            )
        # log(
        #     f'pair {landsat_item.id[:26]} + S2 '
        #     f'{sentinel_item.properties.get("s2:mgrs_tile", "")}: '
        #     f'{len(cells)} candidates -> {accepted_for_pair} accepted'
        # )
        log(
            f'pair {landsat_item.id[:26]} + S2 '
            f'{sentinel_item.properties.get("s2:mgrs_tile", "")}: '
            f'{already_covered} already-covered | '
            f'{len(cells)} fresh candidates -> '
            f'{accepted_for_pair} accepted'
        )

    with phase_done_path.open('a', encoding='utf-8') as stream:
        stream.write(window_start + '\n')
    rej_total.update(rejected)
    REJECTION_COUNTS.write_text(
        json.dumps(dict(rej_total), indent=2), encoding='utf-8'
    )
    LANDSCAPE_COUNTS.write_text(
        json.dumps(dict(landscape_counts), indent=2), encoding='utf-8'
    )
    GEOGRAPHIC_COUNTS.write_text(
        json.dumps(dict(geographic_counts), indent=2), encoding='utf-8'
    )
    PRIORITY_COUNTS.write_text(
        json.dumps(dict(priority_counts), indent=2), encoding='utf-8'
    )
    log(
        f'[window {window_index}] +{accepted_in_window} patches | '
        f'total {total}/{TARGET_PAIRS} | classes: {dict(landscape_counts)} | '
        f'geography: {dict(geographic_counts)} | '
        f'priority: {dict(priority_counts)} | '
        f'rejected: {dict(rejected)}'
    )
    if DASHBOARD_EVERY and total - last_dash >= DASHBOARD_EVERY:
        dashboard()
        last_dash = total

bar.close()
manifest_stream.close()
pool.shutdown()
missing_reserves = missing_landscape_reserves(
    landscape_counts, DEFAULT_LANDSCAPE_MINIMUMS
)
missing_reserves = {
    key: value for key, value in missing_reserves.items() if value > 0
}
log(
    f'DONE: {total}/{TARGET_PAIRS} unique patches. '
    f'Landscape counts: {dict(landscape_counts)}'
)
if missing_reserves:
    print(
        'UNMET LANDSCAPE RESERVES:', missing_reserves,
        '\nAdd source coverage for these strata and resume. The collector did '
        'not replace them with common classes.'
    )
missing_geography = {
    name: max(required - geographic_counts.get(name, 0), 0)
    for name, required in DEFAULT_GEOGRAPHIC_MINIMUMS.items()
}
missing_geography = {
    name: count for name, count in missing_geography.items() if count
}
if missing_geography:
    print('UNMET GEOGRAPHIC RESERVES:', missing_geography)
missing_priority = {
    name: max(required - priority_counts.get(name, 0), 0)
    for name, required in DEFAULT_PRIORITY_REGION_MINIMUMS.items()
}
missing_priority = {
    name: count for name, count in missing_priority.items() if count
}
if missing_priority:
    print('UNMET PRIORITY REGION RESERVES:', missing_priority)

In [ ]:
print("\nCURRENT TOTAL:", total)
print("REMAINING SLOTS:", TARGET_PAIRS - total)

print("\nLANDSCAPE COUNTS")
for name, minimum in DEFAULT_LANDSCAPE_MINIMUMS.items():

    current = landscape_counts.get(name, 0)
    maximum = DEFAULT_LANDSCAPE_MAXIMUMS[name]

    print(
        f"{name:16s}"
        f" current={current:4d}"
        f" min={minimum:4d}"
        f" missing={max(minimum-current, 0):4d}"
        f" max={maximum:4d}"
    )


print("\nGEOGRAPHIC COUNTS")
for name, minimum in DEFAULT_GEOGRAPHIC_MINIMUMS.items():

    current = geographic_counts.get(name, 0)

    print(
        f"{name:16s}"
        f" current={current:4d}"
        f" min={minimum:4d}"
        f" missing={max(minimum-current, 0):4d}"
    )


print("\nPRIORITY COUNTS")
for name, minimum in DEFAULT_PRIORITY_REGION_MINIMUMS.items():

    current = priority_counts.get(name, 0)

    print(
        f"{name:20s}"
        f" current={current:4d}"
        f" min={minimum:4d}"
        f" missing={max(minimum-current, 0):4d}"
    )

In [ ]:
dashboard()

In [ ]:

gallery(4)
verify_inside_india()


In [ ]:

manifest_path = Path(MANIFEST)
if not manifest_path.is_file():
    raise FileNotFoundError('No manifest was produced; inspect the collection log above.')
final = pd.read_csv(manifest_path)
print('Pairs:', len(final))
print('Landsat TIFFs:', len(list((OUT_DIR / 'landsat').glob('*.tif'))))
print('Sentinel TIFFs:', len(list((OUT_DIR / 'sentinel').glob('*.tif'))))
print('Output size (GB):', round(sum(
    path.stat().st_size for path in OUT_DIR.rglob('*') if path.is_file()
) / 2**30, 3))
display(final.landscape_class.value_counts().rename('pairs').to_frame())
display(final.geographic_stratum.value_counts().rename('pairs').to_frame())
display(final.priority_region.value_counts().rename('pairs').to_frame())
missing = missing_landscape_reserves(
    Counter(final.landscape_class), DEFAULT_LANDSCAPE_MINIMUMS
)
missing = {name: count for name, count in missing.items() if count}
if len(final) < TARGET_PAIRS or missing:
    print(
        'Dataset is not release-complete yet.',
        {'target': TARGET_PAIRS, 'collected': len(final),
         'missing_landscape_reserves': missing},
    )
missing_geography = {
    name: max(required - int((final.geographic_stratum == name).sum()), 0)
    for name, required in DEFAULT_GEOGRAPHIC_MINIMUMS.items()
}
missing_geography = {
    name: count for name, count in missing_geography.items() if count
}
if missing_geography:
    print('Geographic coverage still incomplete:', missing_geography)
missing_priority = {
    name: max(required - int((final.priority_region == name).sum()), 0)
    for name, required in DEFAULT_PRIORITY_REGION_MINIMUMS.items()
}
missing_priority = {
    name: count for name, count in missing_priority.items() if count
}
if missing_priority:
    print('Western-Himalaya coverage still incomplete:', missing_priority)
cap_violations = {
    name: int((final.landscape_class == name).sum()) - maximum
    for name, maximum in DEFAULT_LANDSCAPE_MAXIMUMS.items()
    if int((final.landscape_class == name).sum()) > maximum
}
if cap_violations:
    raise RuntimeError(f'Landscape hard-cap violations: {cap_violations}')
display(final.head())
display(FileLink(str(manifest_path)))

## Dataset-paper audit and release manifest

This restart-safe section caches per-pair measurements. It creates a
spatially unique release manifest and deterministic `train`/`val`/`test`
splits from complete 100 km equal-area blocks. Online overlap rejection
prevents repeated 4.8 km footprints before the split is assigned. Raw
collected TIFFs and the original manifest remain untouched.

Registration is a gradient phase-correlation residual-shift **proxy**,
not surveyed GCP RMSE. Primary landscape percentages come from the
Esri/Impact Observatory 2025 annual Sentinel-2 10 m product, and
mountain/alpine strata also use Copernicus DEM GLO-30. Annual land cover
is not an acquisition-time label. ESA WorldCover 2021 is retained only
as a historical comparison.
Climate zones are not invented; join a cited climate raster before making
climate claims.

In [ ]:
"""Paper-ready audits for the Planetary Computer India pair collector.

This module deliberately depends only on packages already used by the standalone
Kaggle notebook.  The notebook builder embeds this file verbatim so an imported
notebook does not need the repository at runtime.
"""
from __future__ import annotations

import hashlib
import json
import math
from collections import Counter
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd

try:
    from .pc_landscape import (
        DEFAULT_LANDSCAPE_MAXIMUMS,
        DEFAULT_LANDSCAPE_MINIMUMS,
        DEFAULT_PRIORITY_REGION_MINIMUMS,
        ESRI_LANDCOVER_COLUMNS,
        LANDSCAPE_CLASSES,
        WORLDCOVER_COLUMNS,
        balanced_spatial_splits,
        split_balance_table,
    )
except (ImportError, ModuleNotFoundError):
    # In the standalone notebook pc_landscape.py is executed in this namespace
    # immediately before this embedded module.
    required = (
        "DEFAULT_LANDSCAPE_MAXIMUMS", "DEFAULT_LANDSCAPE_MINIMUMS",
        "DEFAULT_PRIORITY_REGION_MINIMUMS", "ESRI_LANDCOVER_COLUMNS",
        "LANDSCAPE_CLASSES", "WORLDCOVER_COLUMNS", "balanced_spatial_splits",
        "split_balance_table",
    )
    missing = [name for name in required if name not in globals()]
    if missing:
        raise ImportError(f"Missing embedded landscape helpers: {missing}")


PC_REPORT_VERSION = 2
PC_BANDS = ("blue", "green", "red", "nir")
PC_BAND_DEFINITIONS = (
    {"sensor": "Landsat 8/9 OLI", "name": "blue", "asset": "blue", "approx_um": "0.45-0.51"},
    {"sensor": "Landsat 8/9 OLI", "name": "green", "asset": "green", "approx_um": "0.53-0.59"},
    {"sensor": "Landsat 8/9 OLI", "name": "red", "asset": "red", "approx_um": "0.64-0.67"},
    {"sensor": "Landsat 8/9 OLI", "name": "nir", "asset": "nir08", "approx_um": "0.85-0.88"},
    {"sensor": "Sentinel-2 MSI", "name": "blue", "asset": "B02", "approx_um": "0.46-0.52"},
    {"sensor": "Sentinel-2 MSI", "name": "green", "asset": "B03", "approx_um": "0.54-0.58"},
    {"sensor": "Sentinel-2 MSI", "name": "red", "asset": "B04", "approx_um": "0.65-0.68"},
    {"sensor": "Sentinel-2 MSI", "name": "nir", "asset": "B08", "approx_um": "0.78-0.90"},
)


def _write_table(frame: pd.DataFrame, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    frame.to_csv(path, index=False)
    try:
        path.with_suffix(".tex").write_text(
            frame.to_latex(index=False, float_format="%.4f"), encoding="utf-8"
        )
    except (ImportError, ValueError):
        pass


def _save_figure(figure: Any, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    figure.savefig(path, dpi=300, bbox_inches="tight", facecolor="white")
    figure.savefig(path.with_suffix(".pdf"), bbox_inches="tight", facecolor="white")


def _safe_corr(first: np.ndarray, second: np.ndarray) -> float:
    first = np.asarray(first, dtype=np.float64).ravel()
    second = np.asarray(second, dtype=np.float64).ravel()
    if first.std() < 1e-9 or second.std() < 1e-9:
        return float("nan")
    return float(np.corrcoef(first, second)[0, 1])


def _gradient_image(image: np.ndarray) -> np.ndarray:
    vertical, horizontal = np.gradient(np.asarray(image, dtype=np.float64))
    gradient = np.hypot(horizontal, vertical)
    gradient -= gradient.mean()
    return gradient


def _subpixel_phase_shift(reference: np.ndarray, moving: np.ndarray) -> tuple[float, float, float]:
    """Return dy, dx and a peak-to-background confidence on one common grid."""

    reference = _gradient_image(reference)
    moving = _gradient_image(moving)
    cross = np.fft.fft2(reference) * np.conj(np.fft.fft2(moving))
    cross /= np.maximum(np.abs(cross), 1e-12)
    response = np.abs(np.fft.ifft2(cross))
    peak = np.unravel_index(int(np.argmax(response)), response.shape)
    shifts = []
    for axis, coordinate in enumerate(peak):
        size = response.shape[axis]
        before = list(peak)
        after = list(peak)
        before[axis] = (coordinate - 1) % size
        after[axis] = (coordinate + 1) % size
        left = float(response[tuple(before)])
        center = float(response[peak])
        right = float(response[tuple(after)])
        denominator = left - 2.0 * center + right
        offset = 0.5 * (left - right) / denominator if abs(denominator) > 1e-12 else 0.0
        shift = float(coordinate) + float(np.clip(offset, -0.5, 0.5))
        if shift > size / 2:
            shift -= size
        shifts.append(shift)
    confidence = float(response[peak] / max(np.median(response), 1e-12))
    return shifts[0], shifts[1], confidence


def _pair_metrics(row: Any, root: Path) -> dict[str, Any]:
    import rasterio

    low_path = root / "landsat" / f"{row.pair_id}.tif"
    high_path = root / "sentinel" / f"{row.pair_id}.tif"
    if not low_path.is_file() or not high_path.is_file():
        raise FileNotFoundError(f"Missing TIFF pair for {row.pair_id}")
    with rasterio.open(low_path) as source:
        low_raw = source.read()
        low_shape = (source.count, source.height, source.width)
        low_crs = str(source.crs)
        low_bounds = np.asarray(source.bounds, dtype=np.float64)
        low_resolution = tuple(map(float, source.res))
    with rasterio.open(high_path) as source:
        high_raw = source.read()
        high_shape = (source.count, source.height, source.width)
        high_crs = str(source.crs)
        high_bounds = np.asarray(source.bounds, dtype=np.float64)
        high_resolution = tuple(map(float, source.res))
    if low_shape != (4, 160, 160) or high_shape != (4, 480, 480):
        raise ValueError(f"Unexpected geometry for {row.pair_id}: {low_shape} -> {high_shape}")
    if low_crs != high_crs or not np.allclose(low_bounds, high_bounds, atol=1e-4):
        raise ValueError(f"Geospatial footprint mismatch for {row.pair_id}")

    low = np.clip(low_raw.astype(np.float32) / 10000.0, 0.0, 1.0)
    high = np.clip(high_raw.astype(np.float32) / 10000.0, 0.0, 1.0)
    high_down = high.reshape(4, 160, 3, 160, 3).mean((2, 4))
    low_gray = 0.2126 * low[2] + 0.7152 * low[1] + 0.0722 * low[0]
    high_gray = 0.2126 * high_down[2] + 0.7152 * high_down[1] + 0.0722 * high_down[0]
    shift_y, shift_x, shift_confidence = _subpixel_phase_shift(high_gray, low_gray)
    shift_lr = float(math.hypot(shift_y, shift_x))

    epsilon = 1e-6
    low_ndvi = (low[3] - low[2]) / (low[3] + low[2] + epsilon)
    high_ndvi = (high[3] - high[2]) / (high[3] + high[2] + epsilon)
    low_ndwi = (low[1] - low[3]) / (low[1] + low[3] + epsilon)
    high_ndwi = (high[1] - high[3]) / (high[1] + high[3] + epsilon)
    differences = low - high_down

    result: dict[str, Any] = {
        "report_version": PC_REPORT_VERSION,
        "pair_id": row.pair_id,
        "geometry_valid": True,
        "lr_resolution_x_m": low_resolution[0],
        "hr_resolution_x_m": high_resolution[0],
        "cross_sensor_mae": float(np.abs(differences).mean()),
        "cross_sensor_rmse": float(np.sqrt(np.square(differences).mean())),
        "registration_shift_lr_px": shift_lr,
        "registration_shift_10m_px": shift_lr * 3.0,
        "registration_shift_m": shift_lr * 30.0,
        "registration_confidence": shift_confidence,
        "lr_ndvi_mean": float(low_ndvi.mean()),
        "hr_ndvi_mean": float(high_ndvi.mean()),
        "hr_ndvi_std": float(high_ndvi.std()),
        "lr_ndwi_mean": float(low_ndwi.mean()),
        "hr_ndwi_mean": float(high_ndwi.mean()),
        "hr_rgb_brightness": float(high[:3].mean()),
    }
    for index, band in enumerate(PC_BANDS):
        result[f"lr_{band}_mean"] = float(low[index].mean())
        result[f"lr_{band}_std"] = float(low[index].std())
        result[f"hr_{band}_mean"] = float(high[index].mean())
        result[f"hr_{band}_std"] = float(high[index].std())
        result[f"downsampled_{band}_corr"] = _safe_corr(low[index], high_down[index])
        result[f"downsampled_{band}_mae"] = float(np.abs(differences[index]).mean())
    return result


def _spectral_proxy(row: Any) -> str:
    if row.hr_ndwi_mean > 0.12 and row.hr_ndvi_mean < 0.12:
        return "water_dominant"
    if row.hr_ndvi_mean >= 0.50:
        return "dense_vegetation"
    if row.hr_ndvi_mean >= 0.28:
        return "vegetation"
    if row.hr_rgb_brightness >= 0.22 and row.hr_ndvi_mean < 0.20:
        return "bright_low_vegetation"
    return "mixed_surface"


def _macro_region(state: str) -> str:
    value = str(state).casefold()
    groups = {
        "North": ("delhi", "haryana", "himachal", "jammu", "ladakh", "punjab",
                  "rajasthan", "uttarakhand", "uttar pradesh", "chandigarh"),
        "West": ("gujarat", "maharashtra", "goa", "dadra", "daman"),
        "Central": ("madhya pradesh", "chhattisgarh"),
        "East": ("bihar", "jharkhand", "odisha", "orissa", "west bengal"),
        "Northeast": ("assam", "arunachal", "manipur", "meghalaya", "mizoram",
                      "nagaland", "sikkim", "tripura"),
        "South": ("andhra", "karnataka", "kerala", "tamil", "telangana",
                  "puducherry", "lakshadweep", "andaman"),
    }
    for region, tokens in groups.items():
        if any(token in value for token in tokens):
            return region
    return "Unassigned"


def _add_administrative_regions(frame: pd.DataFrame, adm1: Any | None) -> pd.DataFrame:
    frame = frame.copy()
    if "state_or_ut" not in frame:
        frame["state_or_ut"] = "unavailable"
    else:
        frame["state_or_ut"] = frame.state_or_ut.fillna("unavailable")
    if adm1 is not None and not adm1.empty:
        import geopandas as gpd

        name_column = next(
            (name for name in ("shapeName", "NAME_1", "name", "NAME") if name in adm1.columns),
            None,
        )
        if name_column:
            points = gpd.GeoDataFrame(
                frame[["pair_id"]].copy(),
                geometry=gpd.points_from_xy(frame.center_lon, frame.center_lat),
                crs=4326,
            )
            joined = gpd.sjoin(
                points, adm1[[name_column, "geometry"]].to_crs(4326),
                how="left", predicate="within",
            )
            names = joined.groupby("pair_id")[name_column].first()
            joined_names = frame.pair_id.map(names)
            missing = frame.state_or_ut.isin(("unavailable", "unassigned", ""))
            frame.loc[missing, "state_or_ut"] = joined_names.loc[missing].fillna(
                "unavailable"
            )
    frame["macro_region"] = frame.state_or_ut.map(_macro_region)
    return frame


def _spatial_arbitration(
    frame: pd.DataFrame,
    *,
    block_size_m: float,
    guard_m: float,
    split_seed: int,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    import shapely

    frame = frame.copy().reset_index(drop=True)
    # Spherical cylindrical equal-area coordinates (standard parallel 30 deg).
    # Only relative metres are needed for overlap arbitration and split blocks.
    radius = 6_371_007.181
    cosine_standard_parallel = math.cos(math.radians(30.0))
    longitude = np.radians(frame.center_lon.to_numpy(dtype=np.float64))
    latitude = np.radians(frame.center_lat.to_numpy(dtype=np.float64))
    x = radius * longitude * cosine_standard_parallel
    y = radius * np.sin(latitude) / cosine_standard_parallel
    frame["equal_area_x_m"] = x
    frame["equal_area_y_m"] = y
    half = 2400.0
    footprints = np.asarray(
        [shapely.box(cx - half, cy - half, cx + half, cy + half) for cx, cy in zip(x, y)],
        dtype=object,
    )
    parents = np.arange(len(frame))

    def find(index: int) -> int:
        while parents[index] != index:
            parents[index] = parents[parents[index]]
            index = int(parents[index])
        return index

    def union(first: int, second: int) -> None:
        root_first, root_second = find(first), find(second)
        if root_first != root_second:
            parents[root_second] = root_first

    tree = shapely.STRtree(footprints)
    for first, footprint in enumerate(footprints):
        for second_value in tree.query(footprint):
            second = int(second_value)
            if second <= first:
                continue
            overlap = shapely.area(shapely.intersection(footprint, footprints[second]))
            if overlap / (4800.0 * 4800.0) >= 0.50:
                union(first, second)
    frame["spatial_component"] = [find(index) for index in range(len(frame))]
    frame["quality_rank"] = (
        frame[["corr_red", "corr_nir"]].min(axis=1).fillna(-1.0) * 1000
        + frame.clear_frac.fillna(0.0) * 100
        - frame.time_diff_min.fillna(1e6) / 1000
    )
    ordered = frame.sort_values(
        ["quality_rank", "pair_id"], ascending=[False, True]
    )
    kept_indices = ordered.groupby("spatial_component", sort=False).head(1).index
    frame["duplicate_removed"] = ~frame.index.isin(kept_indices)

    block_x = np.floor(frame.equal_area_x_m / block_size_m).astype(int)
    block_y = np.floor(frame.equal_area_y_m / block_size_m).astype(int)
    frame["spatial_block_id"] = [f"{first}_{second}" for first, second in zip(block_x, block_y)]
    local_x = frame.equal_area_x_m - block_x * block_size_m
    local_y = frame.equal_area_y_m - block_y * block_size_m
    frame["split_guard_removed"] = (
        (local_x < guard_m) | (local_x > block_size_m - guard_m)
        | (local_y < guard_m) | (local_y > block_size_m - guard_m)
    )
    frame["release_included"] = ~(frame.duplicate_removed | frame.split_guard_removed)

    release = frame.loc[frame.release_included].copy()
    if "landscape_class" not in release:
        raise ValueError(
            "manifest.csv has no landscape_class column. Run the Esri 2025 + DEM "
            "balanced collector instead of assigning splits from RGB appearance."
        )
    split_class_column = "landscape_class"
    if "geographic_stratum" in release:
        components = [
            release.landscape_class.astype(str),
            release.geographic_stratum.astype(str),
        ]
        if "priority_region" in release:
            components.append(release.priority_region.astype(str))
        release["split_balance_stratum"] = components[0]
        for component in components[1:]:
            release["split_balance_stratum"] += "__" + component
        frame.loc[release.index, "split_balance_stratum"] = release[
            "split_balance_stratum"
        ]
        split_class_column = "split_balance_stratum"
    assignments = balanced_spatial_splits(
        release,
        block_column="spatial_block_id",
        class_column=split_class_column,
        seed=split_seed,
        fractions={"train": 0.78, "val": 0.11, "test": 0.11},
    )
    frame["split"] = frame.spatial_block_id.map(assignments).fillna("excluded")
    frame.loc[~frame.release_included, "split"] = "excluded"

    kept_for_component = frame.loc[~frame.duplicate_removed].set_index("spatial_component").pair_id
    frame["retained_pair_for_location"] = frame.spatial_component.map(kept_for_component)
    exclusions = frame.loc[~frame.release_included, [
        "pair_id", "retained_pair_for_location", "duplicate_removed",
        "split_guard_removed", "spatial_block_id",
    ]].copy()
    exclusions["reason"] = np.where(
        exclusions.duplicate_removed, "overlapping_geographic_footprint", "spatial_split_boundary_guard"
    )
    return frame, exclusions


def _metric_cache(frame: pd.DataFrame, root: Path, report_root: Path) -> pd.DataFrame:
    from tqdm.auto import tqdm

    cache_path = report_root / "pair_metrics.csv"
    if cache_path.is_file():
        cached = pd.read_csv(cache_path)
        if "report_version" not in cached or not (cached.report_version == PC_REPORT_VERSION).all():
            cached = pd.DataFrame()
    else:
        cached = pd.DataFrame()
    completed = set(cached.pair_id.astype(str)) if not cached.empty else set()
    additions = []
    pending = frame.loc[~frame.pair_id.astype(str).isin(completed)]
    for row in tqdm(pending.itertuples(index=False), total=len(pending), desc="paper metric audit", unit="pair"):
        additions.append(_pair_metrics(row, root))
        if len(additions) >= 25:
            cached = pd.concat([cached, pd.DataFrame(additions)], ignore_index=True)
            cached.to_csv(cache_path, index=False)
            additions.clear()
    if additions:
        cached = pd.concat([cached, pd.DataFrame(additions)], ignore_index=True)
        cached.to_csv(cache_path, index=False)
    return cached.drop_duplicates("pair_id", keep="last")


def _report_inventory(report_root: Path) -> pd.DataFrame:
    rows = []
    for path in sorted(report_root.rglob("*")):
        if path.is_file() and path.name not in {"report_inventory.csv", "paper_reports.zip"}:
            digest = hashlib.sha256(path.read_bytes()).hexdigest()
            rows.append({
                "path": str(path.relative_to(report_root)).replace("\\", "/"),
                "bytes": path.stat().st_size,
                "sha256": digest,
            })
    inventory = pd.DataFrame(rows)
    inventory.to_csv(report_root / "report_inventory.csv", index=False)
    return inventory


def generate_pc_paper_reports(
    output_root: str | Path,
    *,
    adm1: Any | None = None,
    collection_rejections: str | Path | None = None,
    sample_limit: int = 300,
    split_seed: int = 42,
    block_size_m: float = 100_000.0,
    guard_m: float = 0.0,
) -> dict[str, Any]:
    """Generate cached tables, figures, release manifest and a compact report ZIP."""

    import matplotlib.pyplot as plt
    import rasterio
    from PIL import Image

    output_root = Path(output_root)
    report_root = output_root / "paper_reports"
    table_root = report_root / "tables"
    figure_root = report_root / "figures"
    report_root.mkdir(parents=True, exist_ok=True)
    table_root.mkdir(parents=True, exist_ok=True)
    figure_root.mkdir(parents=True, exist_ok=True)
    manifest = output_root / "manifest.csv"
    if not manifest.is_file():
        raise FileNotFoundError(f"Missing collection manifest: {manifest}")
    source = pd.read_csv(manifest)
    if source.empty:
        raise ValueError("Collection manifest is empty")
    if source.pair_id.duplicated().any():
        raise ValueError("pair_id values must be unique before the paper audit")
    required_landscape = {
        "landscape_class", "geographic_stratum", "priority_region",
        "landcover_year", "landcover_source", "worldcover_year", "elevation_p90_m",
        "slope_p90_deg", *ESRI_LANDCOVER_COLUMNS, *WORLDCOVER_COLUMNS,
    }
    missing_landscape = sorted(required_landscape - set(source.columns))
    if missing_landscape:
        raise ValueError(
            "The manifest predates the Esri 2025 + DEM balanced protocol; "
            f"missing columns: {missing_landscape}"
        )

    metrics = _metric_cache(source, output_root, report_root)
    frame = source.merge(metrics, on="pair_id", how="left", validate="one_to_one")
    if frame.report_version.isna().any():
        raise RuntimeError("The metric cache is incomplete")
    frame = _add_administrative_regions(frame, adm1)
    frame["spectral_composition_proxy"] = frame.apply(_spectral_proxy, axis=1)
    frame["quality_flag"] = np.select(
        [
            (frame.clear_frac >= 0.995) & (frame[["corr_red", "corr_nir"]].min(axis=1) >= 0.90),
            (frame.clear_frac >= 0.99) & (frame[["corr_red", "corr_nir"]].min(axis=1) >= 0.80),
        ],
        ["excellent", "good"],
        default="acceptable",
    )
    frame, exclusions = _spatial_arbitration(
        frame, block_size_m=block_size_m, guard_m=guard_m, split_seed=split_seed
    )
    release = frame.loc[frame.release_included].copy()
    if release.empty:
        raise RuntimeError("No pairs remain after spatial arbitration")
    release.to_csv(report_root / "paper_release_manifest.csv", index=False)
    exclusions.to_csv(report_root / "paper_release_exclusions.csv", index=False)
    for split_name in ("train", "val", "test"):
        release.loc[release.split == split_name].to_csv(
            report_root / f"{split_name}.csv", index=False
        )

    split_table = release.groupby("split").size().rename("pairs").reset_index()
    split_table["percentage"] = split_table.pairs / len(release) * 100
    state_table = release.groupby(["macro_region", "state_or_ut"], dropna=False).size().rename("pairs").reset_index()
    region_table = release.groupby("macro_region").size().rename("pairs").reset_index()
    proxy_table = release.groupby("spectral_composition_proxy").size().rename("pairs").reset_index()
    landscape_table = (
        release.groupby("landscape_class").size().rename("pairs").reset_index()
    )
    landscape_table["percentage"] = landscape_table.pairs / len(release) * 100
    landscape_split_table = split_balance_table(release)
    geographic_table = (
        release.groupby("geographic_stratum").size().rename("pairs").reset_index()
    )
    geographic_table["percentage"] = geographic_table.pairs / len(release) * 100
    geographic_split_table = split_balance_table(
        release, class_column="geographic_stratum"
    )
    priority_table = (
        release.groupby("priority_region").size().rename("pairs").reset_index()
    )
    priority_table["percentage"] = priority_table.pairs / len(release) * 100
    priority_split_table = split_balance_table(
        release, class_column="priority_region"
    )
    esri_rows = []
    for column in ESRI_LANDCOVER_COLUMNS:
        values = release[column].astype(float)
        esri_rows.append({
            "esri2025_class": column.removeprefix("esri2025_").removesuffix("_pct"),
            "mean_patch_percent": values.mean(),
            "std_patch_percent": values.std(),
            "p10_patch_percent": values.quantile(0.10),
            "median_patch_percent": values.median(),
            "p90_patch_percent": values.quantile(0.90),
            "patches_with_class": int((values > 0).sum()),
        })
    esri_table = pd.DataFrame(esri_rows)
    worldcover_rows = []
    for column in WORLDCOVER_COLUMNS:
        values = release[column].astype(float)
        worldcover_rows.append({
            "worldcover_class": column.removeprefix("wc2021_").removesuffix("_pct"),
            "mean_patch_percent": values.mean(),
            "std_patch_percent": values.std(),
            "p10_patch_percent": values.quantile(0.10),
            "median_patch_percent": values.median(),
            "p90_patch_percent": values.quantile(0.90),
            "patches_with_class": int((values > 0).sum()),
        })
    worldcover_table = pd.DataFrame(worldcover_rows)
    terrain_table = pd.DataFrame([{
        "pairs": len(release),
        "mean_elevation_m": release.elevation_mean_m.mean(),
        "median_elevation_m": release.elevation_mean_m.median(),
        "p90_elevation_m": release.elevation_p90_m.quantile(0.90),
        "maximum_elevation_m": release.elevation_max_m.max(),
        "mean_slope_deg": release.slope_mean_deg.mean(),
        "p90_slope_deg": release.slope_p90_deg.quantile(0.90),
    }])
    observed_landscapes = Counter(release.landscape_class.astype(str))
    reserve_table = pd.DataFrame([
        {
            "landscape_class": name,
            "collection_minimum": int(required),
            "collection_maximum": int(DEFAULT_LANDSCAPE_MAXIMUMS[name]),
            "release_pairs_after_spatial_audit": int(observed_landscapes.get(name, 0)),
            "release_meets_collection_minimum": int(observed_landscapes.get(name, 0)) >= int(required),
            "release_respects_collection_maximum": (
                int(observed_landscapes.get(name, 0))
                <= int(DEFAULT_LANDSCAPE_MAXIMUMS[name])
            ),
        }
        for name, required in DEFAULT_LANDSCAPE_MINIMUMS.items()
    ])
    priority_reserve_table = pd.DataFrame([
        {
            "priority_region": name,
            "collection_minimum": int(required),
            "release_pairs_after_spatial_audit": int(
                (release.priority_region == name).sum()
            ),
            "release_meets_collection_minimum": int(
                (release.priority_region == name).sum()
            ) >= int(required),
        }
        for name, required in DEFAULT_PRIORITY_REGION_MINIMUMS.items()
    ])
    temporal_table = pd.DataFrame([{
        "pairs": len(release),
        "mean_day_gap": release.day_gap.mean(),
        "maximum_day_gap": release.day_gap.max(),
        "same_day_pairs": int((release.day_gap == 0).sum()),
        "same_day_percent": float((release.day_gap == 0).mean() * 100),
        "mean_time_difference_min": release.time_diff_min.mean(),
        "median_time_difference_min": release.time_diff_min.median(),
        "p95_time_difference_min": release.time_diff_min.quantile(0.95),
        "maximum_time_difference_min": release.time_diff_min.max(),
        "first_date": release.date_L.min(),
        "last_date": release.date_L.max(),
    }])
    registration_table = pd.DataFrame([{
        "audited_pairs": len(release),
        "metric": "gradient phase-correlation residual proxy",
        "mean_shift_10m_px": release.registration_shift_10m_px.mean(),
        "std_shift_10m_px": release.registration_shift_10m_px.std(),
        "median_shift_10m_px": release.registration_shift_10m_px.median(),
        "p95_shift_10m_px": release.registration_shift_10m_px.quantile(0.95),
        "maximum_shift_10m_px": release.registration_shift_10m_px.max(),
        "mean_shift_m": release.registration_shift_m.mean(),
        "p95_shift_m": release.registration_shift_m.quantile(0.95),
    }])
    cloud_table = pd.DataFrame([{
        "mean_joint_cloud_shadow_percent": float((1.0 - release.clear_frac).mean() * 100),
        "maximum_joint_cloud_shadow_percent": float((1.0 - release.clear_frac).max() * 100),
        "pairs_below_1_percent": int(((1.0 - release.clear_frac) <= 0.01).sum()),
        "percent_below_1_percent": float(((1.0 - release.clear_frac) <= 0.01).mean() * 100),
        "mean_landsat_scene_cloud_percent": release.cloud_L.mean(),
        "mean_sentinel_scene_cloud_percent": release.cloud_S.mean(),
    }])
    agreement_table = pd.DataFrame([{
        "mean_cross_sensor_mae": release.cross_sensor_mae.mean(),
        "mean_cross_sensor_rmse": release.cross_sensor_rmse.mean(),
        **{f"mean_{band}_correlation": release[f"downsampled_{band}_corr"].mean()
           for band in PC_BANDS},
    }])
    source_table = release.groupby(["platform", "s2_platform"]).size().rename("pairs").reset_index()
    quality_table = release.groupby("quality_flag").size().rename("pairs").reset_index()
    arbitration_table = pd.DataFrame([{
        "collected_pairs": len(frame),
        "overlapping_footprints_removed": int(frame.duplicate_removed.sum()),
        "split_boundary_guard_removed": int(frame.split_guard_removed.sum()),
        "release_pairs": len(release),
        "unique_spatial_blocks": int(release.spatial_block_id.nunique()),
        "block_size_km": block_size_m / 1000,
        "boundary_guard_km": guard_m / 1000,
    }])
    climate_status = pd.DataFrame([{
        "field": "climate_zone",
        "status": "not_assigned",
        "reason": "No authoritative climate raster is bundled; geographic region is reported instead.",
    }])
    comparison = pd.DataFrame([
        dict(dataset="This work", pairs=len(release), lr_size="160x160", hr_size="480x480",
             scale="3x", bands="RGB+NIR", region="India", temporal_gap="same UTC day",
             georeferenced="yes", quality_masks="implicit in strict filtering"),
        dict(dataset="OLI2MSI", pairs="", lr_size="", hr_size="", scale="3x", bands="",
             region="", temporal_gap="", georeferenced="", quality_masks=""),
        dict(dataset="Land2Sent", pairs="", lr_size="", hr_size="", scale="3x", bands="RGB+NIR",
             region="", temporal_gap="", georeferenced="", quality_masks=""),
        dict(dataset="LSSR", pairs="", lr_size="", hr_size="", scale="", bands="",
             region="", temporal_gap="", georeferenced="", quality_masks=""),
    ])
    for name, table in (
        ("split_summary", split_table),
        ("state_and_macro_region_summary", state_table),
        ("macro_region_summary", region_table),
        ("landscape_summary", landscape_table),
        ("landscape_split_balance", landscape_split_table),
        ("geographic_stratum_summary", geographic_table),
        ("geographic_split_balance", geographic_split_table),
        ("priority_region_summary", priority_table),
        ("priority_region_split_balance", priority_split_table),
        ("esri2025_landcover_percentage_summary", esri_table),
        ("worldcover_percentage_summary", worldcover_table),
        ("terrain_summary", terrain_table),
        ("landscape_reserve_audit", reserve_table),
        ("priority_region_reserve_audit", priority_reserve_table),
        ("spectral_composition_proxy_summary", proxy_table),
        ("temporal_summary", temporal_table),
        ("coregistration_proxy_summary", registration_table),
        ("cloud_quality_summary", cloud_table),
        ("cross_sensor_agreement", agreement_table),
        ("sensor_platform_summary", source_table),
        ("quality_flags", quality_table),
        ("spatial_arbitration", arbitration_table),
        ("climate_zone_status", climate_status),
        ("existing_dataset_comparison_TEMPLATE", comparison),
    ):
        _write_table(table, table_root / f"{name}.csv")

    band_rows = []
    for sensor, prefix in (("Landsat_LR", "lr"), ("Sentinel_HR", "hr")):
        for band in PC_BANDS:
            band_rows.append({
                "sensor": sensor,
                "band": band,
                "mean_reflectance": release[f"{prefix}_{band}_mean"].mean(),
                "between_patch_std": release[f"{prefix}_{band}_mean"].std(),
                "mean_within_patch_std": release[f"{prefix}_{band}_std"].mean(),
            })
    band_table = pd.DataFrame(band_rows)
    _write_table(band_table, table_root / "band_statistics.csv")
    _write_table(pd.DataFrame(PC_BAND_DEFINITIONS), table_root / "band_definitions.csv")

    palette = {
        "North": "#1D4ED8", "West": "#D97706", "Central": "#0F766E",
        "East": "#7C3AED", "Northeast": "#DC2626", "South": "#15803D",
        "Unassigned": "#64748B",
    }
    figure, axis = plt.subplots(figsize=(8.4, 8.0))
    if adm1 is not None:
        adm1.to_crs(4326).boundary.plot(ax=axis, color="#CBD5E1", linewidth=0.35)
    for region, values in release.groupby("macro_region"):
        axis.scatter(values.center_lon, values.center_lat, s=10, alpha=0.38,
                     color=palette.get(region, "#64748B"), edgecolors="none",
                     label=f"{region} ({len(values)})")
    axis.set(xlabel="Longitude", ylabel="Latitude", title="Spatial coverage of release pairs")
    axis.grid(color="#E2E8F0", linewidth=0.5)
    axis.legend(frameon=False, fontsize=8)
    figure.tight_layout(); _save_figure(figure, figure_root / "spatial_coverage.png"); plt.close(figure)

    figure, axes = plt.subplots(1, 2, figsize=(12, 4.8))
    density = axes[0].hexbin(release.center_lon, release.center_lat, gridsize=30,
                             mincnt=1, cmap="viridis", linewidths=0)
    figure.colorbar(density, ax=axes[0], label="Pairs per hexagon")
    axes[0].set(xlabel="Longitude", ylabel="Latitude", title="Spatial density")
    for split, values in release.groupby("split"):
        axes[1].scatter(values.center_lon, values.center_lat, s=9, alpha=0.42,
                        label=f"{split} ({len(values)})")
    axes[1].set(xlabel="Longitude", ylabel="Latitude", title="Leakage-controlled spatial splits")
    axes[1].legend(frameon=False)
    for axis in axes: axis.grid(color="#E2E8F0", linewidth=0.5)
    figure.tight_layout(); _save_figure(figure, figure_root / "spatial_density_and_splits.png"); plt.close(figure)

    figure, axes = plt.subplots(1, 2, figsize=(12, 4.8))
    states = state_table.loc[state_table.state_or_ut != "unavailable"].groupby(
        "state_or_ut"
    ).pairs.sum().sort_values(ascending=False).head(15).sort_values()
    if len(states):
        axes[0].barh(states.index, states.values, color="#0F766E")
        axes[0].set(xlabel="Release pairs", title="Leading states and union territories")
    else:
        axes[0].text(0.5, 0.5, "ADM1 labels unavailable", ha="center", va="center")
        axes[0].set_axis_off()
    contributions = release.groupby("s2_tile").size().sort_values(ascending=False)
    axes[1].bar(range(len(contributions)), contributions.values, color="#D97706")
    axes[1].set(xlabel="Sentinel-2 source tile", ylabel="Release pairs",
                title="Source-tile contribution after arbitration")
    axes[1].tick_params(axis="x", bottom=False, labelbottom=False)
    for axis in axes:
        axis.grid(axis="x" if axis is axes[0] else "y", color="#E2E8F0", linewidth=0.6)
    figure.tight_layout(); _save_figure(figure, figure_root / "regional_and_source_contributions.png"); plt.close(figure)

    figure, axis = plt.subplots(figsize=(7.6, 4.5))
    arbitration_values = [
        len(release), int(frame.duplicate_removed.sum()), int(frame.split_guard_removed.sum())
    ]
    axis.bar(
        ["Release pairs", "Overlapping footprints\nremoved", "Split-boundary guard\nremoved"],
        arbitration_values,
        color=["#0F766E", "#B91C1C", "#D97706"],
    )
    axis.set(ylabel="Pairs", title="Spatial arbitration and leakage control")
    axis.grid(axis="y", color="#E2E8F0", linewidth=0.6)
    figure.tight_layout(); _save_figure(figure, figure_root / "spatial_arbitration.png"); plt.close(figure)

    landscape_palette = {
        "alpine": "#93C5FD", "mountain": "#64748B", "urban": "#DC2626",
        "desert_bare": "#D97706", "forest": "#166534", "cropland": "#84CC16",
        "water_wetland": "#2563EB", "shrub_grass": "#A16207", "mixed": "#7C3AED",
    }
    figure, axes = plt.subplots(1, 2, figsize=(14, 5.6))
    for landscape, values in release.groupby("landscape_class"):
        axes[0].scatter(
            values.center_lon, values.center_lat, s=10, alpha=0.42,
            color=landscape_palette.get(landscape, "#334155"),
            edgecolors="none", label=f"{landscape.replace('_', ' ')} ({len(values)})",
        )
    axes[0].set(
        xlabel="Longitude", ylabel="Latitude",
        title="Esri 2025 annual land cover + DEM sampling strata",
    )
    axes[0].legend(frameon=False, fontsize=7, ncol=2)
    split_landscape = pd.crosstab(release.split, release.landscape_class).reindex(
        index=["train", "val", "test"], columns=LANDSCAPE_CLASSES, fill_value=0
    )
    split_landscape.plot(
        kind="bar", stacked=True, ax=axes[1],
        color=[landscape_palette[name] for name in LANDSCAPE_CLASSES], width=0.72,
    )
    axes[1].set(
        xlabel="Split", ylabel="Pairs",
        title="Landscape composition of leakage-controlled splits",
    )
    axes[1].tick_params(axis="x", rotation=0)
    axes[1].legend(frameon=False, fontsize=7, ncol=2, title=None)
    for axis in axes:
        axis.grid(color="#E2E8F0", linewidth=0.5)
    figure.tight_layout()
    _save_figure(figure, figure_root / "landscape_coverage_and_split_balance.png")
    plt.close(figure)

    figure, axes = plt.subplots(1, 2, figsize=(12, 4.7))
    order = [name for name in LANDSCAPE_CLASSES if name in set(release.landscape_class)]
    elevation_values = [
        release.loc[release.landscape_class == name, "elevation_mean_m"].to_numpy()
        for name in order
    ]
    axes[0].boxplot(elevation_values, tick_labels=[name.replace("_", " ") for name in order])
    axes[0].tick_params(axis="x", rotation=35)
    axes[0].set(ylabel="Mean elevation (m)", title="Elevation by sampling stratum")
    cover_means = esri_table.sort_values("mean_patch_percent", ascending=True)
    axes[1].barh(
        cover_means.esri2025_class.str.replace("_", " "),
        cover_means.mean_patch_percent, color="#0F766E",
    )
    axes[1].set(xlabel="Mean patch coverage (%)", title="Esri 2025 annual composition")
    for axis in axes:
        axis.grid(color="#E2E8F0", linewidth=0.5)
    figure.tight_layout()
    _save_figure(figure, figure_root / "terrain_and_landcover_distribution.png")
    plt.close(figure)

    figure, axis = plt.subplots(figsize=(7.2, 5.8))
    proxy_values = proxy_table.sort_values("pairs", ascending=False)
    colors = ["#0F766E", "#15803D", "#D97706", "#1D4ED8", "#64748B"][:len(proxy_values)]
    axis.pie(proxy_values.pairs, labels=proxy_values.spectral_composition_proxy.str.replace("_", " "),
             autopct="%1.1f%%", startangle=90, colors=colors,
             wedgeprops={"linewidth": 1.2, "edgecolor": "white"})
    axis.set_title("Secondary RGB+NIR spectral proxy (diagnostic only)")
    figure.tight_layout(); _save_figure(figure, figure_root / "spectral_composition_proxy_pie.png"); plt.close(figure)

    dates = pd.to_datetime(release.date_L)
    figure, axes = plt.subplots(1, 3, figsize=(15, 4.3))
    months = dates.dt.month.value_counts().reindex(range(1, 13), fill_value=0)
    axes[0].bar(months.index, months.values, color="#0F766E")
    axes[0].set(xlabel="Month", ylabel="Pairs", title="Acquisition month distribution", xticks=range(1, 13))
    axes[1].hist(release.time_diff_min, bins=24, color="#D97706", edgecolor="white")
    axes[1].set(xlabel="Acquisition-time difference (minutes)", ylabel="Pairs", title="Same-day time separation")
    gaps = np.sort(release.time_diff_min.to_numpy(dtype=float))
    axes[2].step(gaps, np.arange(1, len(gaps) + 1) / len(gaps), where="post", color="#1D4ED8", linewidth=2)
    axes[2].set(xlabel="Acquisition-time difference (minutes)", ylabel="Cumulative fraction",
                title="Temporal-gap cumulative distribution", ylim=(0, 1.02))
    for axis in axes: axis.grid(color="#E2E8F0", linewidth=0.6)
    figure.tight_layout(); _save_figure(figure, figure_root / "temporal_distribution.png"); plt.close(figure)

    figure, axes = plt.subplots(1, 3, figsize=(15, 4.3))
    axes[0].hist(release.registration_shift_10m_px, bins=24, color="#334155", edgecolor="white")
    axes[0].axvline(1.0, color="#DC2626", linestyle="--", label="1 target pixel")
    axes[0].set(xlabel="Residual shift proxy (10 m pixels)", ylabel="Pairs", title="Cross-sensor alignment proxy")
    axes[0].legend(frameon=False)
    axes[1].hist((1.0 - release.clear_frac) * 100, bins=20, color="#15803D", edgecolor="white")
    axes[1].set(xlabel="Joint cloud + shadow (%)", ylabel="Pairs", title="Patch-level obstruction")
    axes[2].hist(release[["corr_red", "corr_nir"]].min(axis=1), bins=20,
                 color="#7C3AED", edgecolor="white")
    axes[2].set(xlabel="Minimum red/NIR correlation", ylabel="Pairs", title="Cross-sensor structural agreement")
    for axis in axes: axis.grid(color="#E2E8F0", linewidth=0.6)
    figure.tight_layout(); _save_figure(figure, figure_root / "quality_distributions.png"); plt.close(figure)

    figure, axes = plt.subplots(1, 2, figsize=(12, 4.6))
    regions = sorted(release.macro_region.unique())
    values = [release.loc[release.macro_region == region, "hr_ndvi_mean"].to_numpy() for region in regions]
    axes[0].boxplot(values, patch_artist=True,
                    boxprops={"facecolor": "#BBF7D0", "edgecolor": "#15803D"},
                    medianprops={"color": "#92400E", "linewidth": 1.4})
    axes[0].set_xticks(range(1, len(regions) + 1), regions)
    axes[0].tick_params(axis="x", rotation=25)
    axes[0].set(ylabel="Mean target NDVI", title="Spectral diversity by macro-region")
    axes[1].scatter(dates.dt.month, release.hr_ndvi_mean, s=9, alpha=0.25, color="#0F766E")
    axes[1].set(xlabel="Acquisition month", ylabel="Mean target NDVI",
                title="Seasonal vegetation distribution", xticks=range(1, 13))
    for axis in axes: axis.grid(color="#E2E8F0", linewidth=0.6)
    figure.tight_layout(); _save_figure(figure, figure_root / "ndvi_diversity.png"); plt.close(figure)

    sampled = release.sort_values("pair_id").iloc[: min(sample_limit, len(release))]
    spectra = []
    for row in sampled.itertuples(index=False):
        with rasterio.open(output_root / "sentinel" / f"{row.pair_id}.tif") as source_file:
            image = np.clip(source_file.read()[:, ::24, ::24].astype(np.float32) / 10000.0, 0, 1)
        spectra.append(image.reshape(4, -1).T)
    spectral_samples = np.concatenate(spectra, axis=0)
    correlation = np.corrcoef(spectral_samples, rowvar=False)
    correlation_table = pd.DataFrame(correlation, index=PC_BANDS, columns=PC_BANDS)
    correlation_table.to_csv(table_root / "sentinel_band_correlation.csv")
    figure, axes = plt.subplots(1, 2, figsize=(11.5, 4.8))
    heatmap = axes[0].imshow(correlation, cmap="RdBu_r", vmin=-1, vmax=1)
    axes[0].set_xticks(range(4), PC_BANDS); axes[0].set_yticks(range(4), PC_BANDS)
    for row_index in range(4):
        for column_index in range(4):
            axes[0].text(column_index, row_index, f"{correlation[row_index, column_index]:.2f}",
                         ha="center", va="center", fontsize=9)
    axes[0].set_title("Sentinel-2 target band correlation")
    figure.colorbar(heatmap, ax=axes[0], fraction=0.046, pad=0.04)
    for sensor, values in band_table.groupby("sensor"):
        axes[1].plot(values.band, values.mean_reflectance, marker="o", label=sensor)
    axes[1].set(xlabel="Band", ylabel="Mean physical reflectance", title="Band statistics")
    axes[1].grid(color="#E2E8F0", linewidth=0.6); axes[1].legend(frameon=False)
    figure.tight_layout(); _save_figure(figure, figure_root / "band_statistics_and_correlation.png"); plt.close(figure)

    examples = release.sort_values(["landscape_class", "pair_id"]).groupby(
        "landscape_class", as_index=False
    ).first()
    examples = examples.iloc[: min(9, len(examples))]
    figure, axes = plt.subplots(len(examples), 4, figsize=(13.5, 3.1 * len(examples)), squeeze=False)
    for axis_row, row in zip(axes, examples.itertuples(index=False)):
        with rasterio.open(output_root / "landsat" / f"{row.pair_id}.tif") as source_file:
            low = np.clip(source_file.read().astype(np.float32) / 10000.0, 0, 1)
        with rasterio.open(output_root / "sentinel" / f"{row.pair_id}.tif") as source_file:
            high = np.clip(source_file.read().astype(np.float32) / 10000.0, 0, 1)
        low_rgb = np.clip(low[[2, 1, 0]].transpose(1, 2, 0) / 0.30, 0, 1) ** (1 / 1.2)
        high_rgb = np.clip(high[[2, 1, 0]].transpose(1, 2, 0) / 0.30, 0, 1) ** (1 / 1.2)
        bicubic = np.asarray(Image.fromarray((low_rgb * 255).astype(np.uint8)).resize((480, 480), Image.Resampling.BICUBIC)) / 255.0
        difference = np.abs(bicubic - high_rgb).mean(axis=2)
        ndvi = (high[3] - high[2]) / (high[3] + high[2] + 1e-6)
        contents = (
            (low_rgb, f"{row.landscape_class.replace('_', ' ')}: Landsat 30 m", None),
            (high_rgb, "Sentinel-2 10 m", None),
            (difference, "Bicubic-target RGB difference", "magma"),
            (ndvi, "Sentinel-2 NDVI", "RdYlGn"),
        )
        for axis, (content, title, cmap) in zip(axis_row, contents):
            axis.imshow(content, cmap=cmap, vmin=-1 if title.endswith("NDVI") else None,
                        vmax=1 if title.endswith("NDVI") else None, interpolation="nearest")
            axis.set_title(title, fontsize=9); axis.axis("off")
    figure.tight_layout(); _save_figure(figure, figure_root / "regional_pair_examples.png"); plt.close(figure)

    excluded_examples = exclusions.sort_values(["reason", "pair_id"]).groupby(
        "reason", as_index=False
    ).first().iloc[:4]
    if len(excluded_examples):
        figure, axes = plt.subplots(
            len(excluded_examples), 2,
            figsize=(7.8, 3.1 * len(excluded_examples)), squeeze=False,
        )
        for axis_row, row in zip(axes, excluded_examples.itertuples(index=False)):
            with rasterio.open(output_root / "landsat" / f"{row.pair_id}.tif") as source_file:
                low = np.clip(source_file.read().astype(np.float32) / 10000.0, 0, 1)
            with rasterio.open(output_root / "sentinel" / f"{row.pair_id}.tif") as source_file:
                high = np.clip(source_file.read().astype(np.float32) / 10000.0, 0, 1)
            low_rgb = np.clip(low[[2, 1, 0]].transpose(1, 2, 0) / 0.30, 0, 1) ** (1 / 1.2)
            high_rgb = np.clip(high[[2, 1, 0]].transpose(1, 2, 0) / 0.30, 0, 1) ** (1 / 1.2)
            axis_row[0].imshow(low_rgb, interpolation="nearest")
            axis_row[0].set_title("Excluded Landsat candidate", fontsize=9)
            axis_row[1].imshow(high_rgb, interpolation="nearest")
            axis_row[1].set_title(str(row.reason).replace("_", " "), fontsize=9)
            for axis in axis_row:
                axis.axis("off")
        figure.tight_layout(); _save_figure(figure, figure_root / "excluded_pair_examples.png"); plt.close(figure)

    rejection_counts = Counter(exclusions.reason.astype(str))
    if collection_rejections and Path(collection_rejections).is_file():
        payload = json.loads(Path(collection_rejections).read_text(encoding="utf-8"))
        rejection_counts.update({str(key): int(value) for key, value in payload.items()})
    rejection_table = pd.DataFrame([
        {"reason": reason, "candidates": count}
        for reason, count in sorted(rejection_counts.items(), key=lambda item: -item[1])
    ])
    _write_table(rejection_table, table_root / "rejection_reasons.csv")
    if not rejection_table.empty:
        figure, axis = plt.subplots(figsize=(8.6, max(4.2, 0.34 * len(rejection_table))))
        axis.barh(rejection_table.reason.str.replace("_", " "), rejection_table.candidates, color="#B91C1C")
        axis.invert_yaxis(); axis.set(xlabel="Candidates", title="Collection and release exclusions")
        axis.grid(axis="x", color="#E2E8F0", linewidth=0.6)
        figure.tight_layout(); _save_figure(figure, figure_root / "rejection_reasons.png"); plt.close(figure)

    summary = {
        "report_version": PC_REPORT_VERSION,
        "collected_pairs": int(len(frame)),
        "release_pairs": int(len(release)),
        "split_counts": {str(key): int(value) for key, value in release.split.value_counts().items()},
        "same_day_percent": float((release.day_gap == 0).mean() * 100),
        "mean_time_difference_min": float(release.time_diff_min.mean()),
        "states_and_uts": int(release.loc[release.state_or_ut != "unavailable", "state_or_ut"].nunique()),
        "macro_regions": {str(key): int(value) for key, value in release.macro_region.value_counts().items()},
        "landscape_classes": {
            str(key): int(value)
            for key, value in release.landscape_class.value_counts().items()
        },
        "geographic_strata": {
            str(key): int(value)
            for key, value in release.geographic_stratum.value_counts().items()
        },
        "priority_regions": {
            str(key): int(value)
            for key, value in release.priority_region.value_counts().items()
        },
        "landcover_source": "Esri Impact Observatory Sentinel-2 10m Annual LULC",
        "landcover_year": 2025,
        "worldcover_year": 2021,
        "spectral_composition_proxies": {str(key): int(value) for key, value in release.spectral_composition_proxy.value_counts().items()},
        "mean_joint_cloud_shadow_percent": float((1 - release.clear_frac).mean() * 100),
        "mean_registration_shift_10m_px": float(release.registration_shift_10m_px.mean()),
        "p95_registration_shift_10m_px": float(release.registration_shift_10m_px.quantile(0.95)),
        "mean_cross_sensor_mae": float(release.cross_sensor_mae.mean()),
        "registration_caveat": "Gradient phase-correlation is an image-based residual-shift proxy, not surveyed GCP RMSE.",
        "composition_caveat": (
            "Esri/Impact Observatory 2025 annual land cover is the primary sampling "
            "metadata for 2025 acquisitions, not exact acquisition-date ground truth. "
            "ESA WorldCover 2021 is retained only as a historical comparison. "
            "Mountain/alpine strata additionally use Copernicus DEM GLO-30, a DSM."
        ),
        "climate_caveat": "No climate zone is assigned without joining an authoritative climate product.",
        "spatial_arbitration_caveat": "Repeat coverage is grouped with 4.8 km equal-area footprint proxies at >=50% overlap.",
        "reflectance_protocol": "Landsat C2 L2 and Sentinel-2 L2A surface reflectance, clipped to [0,1] for analysis; source TIFF values remain reflectance x10000.",
    }
    (report_root / "paper_statistics.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
    readme = f"""# Planetary Computer India dataset paper outputs

- Collected pairs: {len(frame)}
- Spatially unique, split-safe release pairs: {len(release)}
- Split counts: {summary['split_counts']}
- Mean acquisition-time separation: {summary['mean_time_difference_min']:.2f} minutes
- Same UTC date: {summary['same_day_percent']:.2f}%

The registration value is a gradient phase-correlation residual proxy, not
independently surveyed GCP RMSE. Sampling strata use the Esri/Impact Observatory
2025 annual Sentinel-2 land-cover map plus Copernicus DEM GLO-30. Annual labels are
not exact acquisition-date ground truth. ESA WorldCover 2021 remains a historical
comparison only. Climate zones remain unassigned until an authoritative climate
raster is joined. Literature comparison fields are intentionally blank until values
are verified against cited dataset versions.
"""
    (report_root / "README.md").write_text(readme, encoding="utf-8")
    inventory = _report_inventory(report_root)

    import zipfile
    archive = output_root / "paper_reports.zip"
    temporary = archive.with_suffix(".tmp.zip")
    with zipfile.ZipFile(temporary, "w", zipfile.ZIP_DEFLATED, allowZip64=True) as destination:
        for path in sorted(report_root.rglob("*")):
            if path.is_file():
                destination.write(path, Path("paper_reports") / path.relative_to(report_root))
    temporary.replace(archive)
    with zipfile.ZipFile(archive) as source_zip:
        if source_zip.testzip() is not None:
            raise RuntimeError("Paper report ZIP verification failed")
    summary["report_files"] = int(len(inventory))
    summary["report_root"] = str(report_root)
    summary["report_archive"] = str(archive)
    return summary


def show_pc_release_pair(
    output_root: str | Path,
    index: int = 0,
    split: str = "test",
) -> dict[str, Any]:
    """Display one indexed release pair with RGB, false colour, NDVI and metadata."""

    import matplotlib.pyplot as plt
    import rasterio

    output_root = Path(output_root)
    manifest = pd.read_csv(output_root / "paper_reports" / "paper_release_manifest.csv")
    rows = manifest.loc[manifest.split == split].reset_index(drop=True)
    if rows.empty:
        raise ValueError(f"No release rows for split={split}")
    row = rows.iloc[int(index) % len(rows)]
    with rasterio.open(output_root / "landsat" / f"{row.pair_id}.tif") as source:
        low = np.clip(source.read().astype(np.float32) / 10000.0, 0, 1)
        low_geo = (str(source.crs), tuple(source.bounds), source.res, source.dtypes)
    with rasterio.open(output_root / "sentinel" / f"{row.pair_id}.tif") as source:
        high = np.clip(source.read().astype(np.float32) / 10000.0, 0, 1)
        high_geo = (str(source.crs), tuple(source.bounds), source.res, source.dtypes)
    low_rgb = np.clip(low[[2, 1, 0]].transpose(1, 2, 0) / 0.30, 0, 1) ** (1 / 1.2)
    high_rgb = np.clip(high[[2, 1, 0]].transpose(1, 2, 0) / 0.30, 0, 1) ** (1 / 1.2)
    false_colour = np.clip(high[[3, 2, 1]].transpose(1, 2, 0) / 0.30, 0, 1) ** (1 / 1.2)
    low_ndvi = (low[3] - low[2]) / (low[3] + low[2] + 1e-6)
    high_ndvi = (high[3] - high[2]) / (high[3] + high[2] + 1e-6)
    figure, axes = plt.subplots(1, 5, figsize=(18, 4.2))
    panels = (
        (low_rgb, "Landsat RGB 30 m", None),
        (high_rgb, "Sentinel-2 RGB 10 m", None),
        (false_colour, "Sentinel-2 NIR-R-G", None),
        (low_ndvi, "Landsat NDVI", "RdYlGn"),
        (high_ndvi, "Sentinel-2 NDVI", "RdYlGn"),
    )
    for axis, (content, title, cmap) in zip(axes, panels):
        axis.imshow(content, cmap=cmap, vmin=-1 if cmap else None, vmax=1 if cmap else None,
                    interpolation="nearest")
        axis.set_title(title); axis.axis("off")
    figure.suptitle(
        f"{split}/{index} | {row.macro_region} | {row.state_or_ut} | "
        f"gap={row.time_diff_min:.1f} min | shift proxy={row.registration_shift_10m_px:.2f} px"
    )
    figure.tight_layout(); plt.show()
    metadata = {
        "pair_id": row.pair_id,
        "landscape_class": row.landscape_class,
        "priority_region": row.priority_region,
        "landcover_year": int(row.landcover_year),
        "worldcover_year": int(row.worldcover_year),
        "elevation_p90_m": float(row.elevation_p90_m),
        "slope_p90_deg": float(row.slope_p90_deg),
        "spectral_composition_proxy": row.spectral_composition_proxy,
        "quality_flag": row.quality_flag,
        "clear_fraction": float(row.clear_frac),
        "red_correlation": float(row.corr_red),
        "nir_correlation": float(row.corr_nir),
        "registration_shift_10m_px": float(row.registration_shift_10m_px),
        "landsat_geospatial": str(low_geo),
        "sentinel_geospatial": str(high_geo),
    }
    metadata.update({column: float(getattr(row, column)) for column in ESRI_LANDCOVER_COLUMNS})
    metadata.update({column: float(getattr(row, column)) for column in WORLDCOVER_COLUMNS})
    from IPython.display import display
    display(pd.DataFrame([metadata]))
    return metadata

In [ ]:
PAPER_SUMMARY = generate_pc_paper_reports(
    OUT_DIR,
    adm1=adm1,
    collection_rejections=REJECTION_COUNTS,
    sample_limit=300,
    split_seed=SEED,
    block_size_m=100_000,
    guard_m=0,
)
print(json.dumps(PAPER_SUMMARY, indent=2))

PAPER_ROOT = OUT_DIR / 'paper_reports'
for table_name in (
    'split_summary.csv',
    'landscape_summary.csv',
    'landscape_split_balance.csv',
    'geographic_stratum_summary.csv',
    'geographic_split_balance.csv',
    'priority_region_summary.csv',
    'priority_region_split_balance.csv',
    'priority_region_reserve_audit.csv',
    'esri2025_landcover_percentage_summary.csv',
    'worldcover_percentage_summary.csv',
    'terrain_summary.csv',
    'landscape_reserve_audit.csv',
    'state_and_macro_region_summary.csv',
    'temporal_summary.csv',
    'coregistration_proxy_summary.csv',
    'cloud_quality_summary.csv',
    'band_statistics.csv',
    'cross_sensor_agreement.csv',
    'spatial_arbitration.csv',
):
    print(f'\n{table_name}')
    display(pd.read_csv(PAPER_ROOT / 'tables' / table_name))

for figure_path in sorted((PAPER_ROOT / 'figures').glob('*.png')):
    display(figure_path.name)
    display(Image(filename=str(figure_path)))

## Indexed release-pair inspection

Change `index` and `split` to inspect any fixed release sample. The RGB
stretch is display-only; GeoTIFF values and metrics use physical surface
reflectance scaled by 10,000 and clipped to `[0,1]` only during analysis.

In [ ]:
show_pc_release_pair(OUT_DIR, index=0, split='test')

## Downloadable outputs

`paper_reports.zip` contains CSV/LaTeX tables, 300-dpi PNG/PDF figures,
release and exclusion manifests, caveats, statistics, and a SHA-256
inventory. Full TIFF archiving remains optional because it temporarily
duplicates the dataset's disk usage.

In [ ]:
paper_archive = OUT_DIR / 'paper_reports.zip'
display(FileLink(str(paper_archive)))
CREATE_ARCHIVE=True
if CREATE_ARCHIVE:
    archive = OUT_DIR.parent / 'SR_pairs_india_ESRI2025_4800_balanced.zip'
    required = sum(path.stat().st_size for path in OUT_DIR.rglob('*') if path.is_file())
    free = shutil.disk_usage(OUT_DIR.parent).free
    if free < required + 512 * 2**20:
        raise OSError(
            f'Need about {(required + 512 * 2**20) / 2**30:.2f} GB free to create ZIP; '
            f'only {free / 2**30:.2f} GB is available. Save the folder as Kaggle output instead.'
        )
    temporary = archive.with_suffix('.tmp.zip')
    with zipfile.ZipFile(temporary, 'w', zipfile.ZIP_STORED, allowZip64=True) as zf:
        for path in sorted(OUT_DIR.rglob('*')):
            if path.is_file():
                zf.write(path, Path(OUT_DIR.name) / path.relative_to(OUT_DIR))
    temporary.replace(archive)
    with zipfile.ZipFile(archive) as zf:
        if zf.testzip() is not None:
            raise RuntimeError('Output ZIP verification failed')
    display(FileLink(str(archive)))
else:
    print('CREATE_ARCHIVE=False: use Save Version -> Save output for the TIFF dataset.')

In [ ]:
from __future__ import annotations

import os
import tempfile
from functools import lru_cache
from pathlib import Path

import gradio as gr
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import rasterio
from PIL import Image, ImageDraw

DATA_DIR = Path(os.environ.get(
    "SR_DATA_DIR",
    "/kaggle/working/SR_pairs_india_ESRI2025_4800_balanced",
))
LANDSAT_DIR = DATA_DIR / "landsat"
SENTINEL_DIR = DATA_DIR / "sentinel"
MANIFEST = (
    DATA_DIR / "manifest_with_dynamicworld.csv"
    if (DATA_DIR / "manifest_with_dynamicworld.csv").is_file()
    else DATA_DIR / "manifest.csv"
)

if not MANIFEST.is_file():
    raise FileNotFoundError(f"Manifest not found: {MANIFEST}\nSet SR_DATA_DIR to your dataset folder.")

DF = pd.read_csv(MANIFEST)
DF["pair_id"] = DF["pair_id"].astype(str)
for c in ("date_L", "date_S"):
    if c in DF:
        DF[c] = pd.to_datetime(DF[c], errors="coerce")
for c in [
    "cloud_L", "cloud_S", "clear_frac", "water_frac", "corr_red", "corr_nir",
    "nir_std", "time_diff_min", "center_lon", "center_lat", "elevation_p90_m",
    "slope_p90_deg",
]:
    if c in DF:
        DF[c] = pd.to_numeric(DF[c], errors="coerce")
if "date_L" in DF:
    DF["month"] = DF["date_L"].dt.month

TABLE_COLUMNS = [
    "pair_id", "date_L", "date_S", "landscape_class", "geographic_stratum",
    "state_or_ut", "priority_region", "platform", "s2_tile", "clear_frac",
    "cloud_L", "cloud_S", "corr_red", "corr_nir", "water_frac", "nir_std",
    "time_diff_min", "elevation_p90_m", "slope_p90_deg",
]


def choices(col):
    if col not in DF:
        return []
    return sorted(DF[col].dropna().astype(str).unique().tolist())


def placeholder(text):
    im = Image.new("RGB", (640, 420), "white")
    ImageDraw.Draw(im).multiline_text((24, 24), text, fill="black", spacing=8)
    return im


@lru_cache(maxsize=96)
def read_pair(pid):
    lp = LANDSAT_DIR / f"{pid}.tif"
    sp = SENTINEL_DIR / f"{pid}.tif"
    if not lp.is_file() or not sp.is_file():
        raise FileNotFoundError(f"Missing TIFF for {pid}\nLandsat: {lp}\nSentinel: {sp}")
    with rasterio.open(lp) as src:
        lr = src.read()
    with rasterio.open(sp) as src:
        hr = src.read()
    return lr, hr


def limits(ref, mode, bands):
    x = ref[list(bands)].astype(np.float32)
    if mode == "Fixed 0–3000":
        return 0.0, 3000.0
    if mode == "Full 0–10000":
        return 0.0, 10000.0
    x = x[np.isfinite(x)]
    if not x.size:
        return 0.0, 10000.0
    lo, hi = np.percentile(x, [2, 98])
    return (float(lo), float(hi)) if hi > lo else (0.0, 10000.0)


def u8(x):
    return np.clip(np.rint(x * 255), 0, 255).astype(np.uint8)


def render(arr, ref, mode, stretch):
    arr = arr.astype(np.float32)
    if mode == "RGB":
        idx = [2, 1, 0]
        lo, hi = limits(ref, stretch, idx)
        x = arr[idx].transpose(1, 2, 0)
        x = np.clip((x - lo) / max(hi - lo, 1e-6), 0, 1)
        return Image.fromarray(u8(x), "RGB")
    if mode == "False color (NIR-R-G)":
        idx = [3, 2, 1]
        lo, hi = limits(ref, stretch, idx)
        x = arr[idx].transpose(1, 2, 0)
        x = np.clip((x - lo) / max(hi - lo, 1e-6), 0, 1)
        return Image.fromarray(u8(x), "RGB")
    if mode == "NIR":
        lo, hi = limits(ref, stretch, [3])
        x = np.clip((arr[3] - lo) / max(hi - lo, 1e-6), 0, 1)
        return Image.fromarray(u8(x), "L").convert("RGB")
    red, nir = arr[2], arr[3]
    ndvi = (nir - red) / (nir + red + 1e-6)
    return Image.fromarray(u8(np.clip((ndvi + 1) / 2, 0, 1)), "L").convert("RGB")


def rgb_ref(hr):
    return render(hr, hr, "RGB", "Auto 2–98%")


def filter_df(search, landscape, geo, priority, states, platforms, months,
              start_date, end_date, min_clear, max_cloud_l, max_cloud_s,
              min_red, min_nir, max_water, max_gap, sort_mode):
    f = DF.copy()
    q = (search or "").strip()
    if q:
        mask = pd.Series(False, index=f.index)
        for c in ["pair_id", "landsat_id", "sentinel_id", "s2_tile", "state_or_ut", "landscape_class"]:
            if c in f:
                mask |= f[c].astype(str).str.contains(q, case=False, na=False, regex=False)
        f = f[mask]
    for col, vals in [
        ("landscape_class", landscape), ("geographic_stratum", geo),
        ("priority_region", priority), ("state_or_ut", states), ("platform", platforms),
    ]:
        if vals and col in f:
            f = f[f[col].astype(str).isin(set(map(str, vals)))]
    if months and "month" in f:
        f = f[f["month"].isin([int(x) for x in months])]
    if "date_L" in f:
        d0 = pd.to_datetime(start_date, errors="coerce") if start_date else pd.NaT
        d1 = pd.to_datetime(end_date, errors="coerce") if end_date else pd.NaT
        if pd.notna(d0): f = f[f.date_L >= d0]
        if pd.notna(d1): f = f[f.date_L < d1 + pd.Timedelta(days=1)]
    checks = [
        ("clear_frac", ">=", min_clear), ("cloud_L", "<=", max_cloud_l),
        ("cloud_S", "<=", max_cloud_s), ("corr_red", ">=", min_red),
        ("corr_nir", ">=", min_nir), ("water_frac", "<=", max_water),
        ("time_diff_min", "<=", max_gap),
    ]
    for col, op, value in checks:
        if col not in f: continue
        s = pd.to_numeric(f[col], errors="coerce")
        f = f[s >= value] if op == ">=" else f[s <= value]
    if sort_mode == "Newest first" and "date_L" in f:
        f = f.sort_values("date_L", ascending=False)
    elif sort_mode == "Lowest correlation first" and {"corr_red", "corr_nir"}.issubset(f.columns):
        f = f.assign(_q=f[["corr_red", "corr_nir"]].min(axis=1)).sort_values("_q").drop(columns="_q")
    elif sort_mode == "Highest correlation first" and {"corr_red", "corr_nir"}.issubset(f.columns):
        f = f.assign(_q=f[["corr_red", "corr_nir"]].min(axis=1)).sort_values("_q", ascending=False).drop(columns="_q")
    elif sort_mode == "Highest clear fraction" and "clear_frac" in f:
        f = f.sort_values("clear_frac", ascending=False)
    elif sort_mode == "Largest time gap" and "time_diff_min" in f:
        f = f.sort_values("time_diff_min", ascending=False)
    elif "date_L" in f:
        f = f.sort_values("date_L")
    return f.reset_index(drop=True)


def table_for(f, n):
    cols = [c for c in TABLE_COLUMNS if c in f]
    out = f[cols].head(int(n)).copy()
    for c in ("date_L", "date_S"):
        if c in out:
            out[c] = pd.to_datetime(out[c], errors="coerce").dt.strftime("%Y-%m-%d")
    return out


def summary(f):
    n = len(f)
    mean_clear = f.clear_frac.mean() if n and "clear_frac" in f else np.nan
    mean_corr = f[[c for c in ["corr_red", "corr_nir"] if c in f]].min(axis=1).mean() if n and any(c in f for c in ["corr_red", "corr_nir"]) else np.nan
    states = f.state_or_ut.nunique() if n and "state_or_ut" in f else 0
    classes = f.landscape_class.nunique() if n and "landscape_class" in f else 0
    def v(x, d=3): return "—" if pd.isna(x) else f"{x:.{d}f}"
    return f'''<div class="kpis">
      <div><b>{n:,}</b><span>Filtered pairs</span></div>
      <div><b>{100*n/max(len(DF),1):.1f}%</b><span>Of dataset</span></div>
      <div><b>{v(mean_clear,4)}</b><span>Mean clear</span></div>
      <div><b>{v(mean_corr,3)}</b><span>Mean min correlation</span></div>
      <div><b>{states}</b><span>States / UTs</span></div>
      <div><b>{classes}</b><span>Classes</span></div>
    </div>'''


def empty_fig(title):
    fig = go.Figure()
    fig.update_layout(title=title, annotations=[dict(text="No matching rows", x=.5, y=.5, xref="paper", yref="paper", showarrow=False)])
    return fig


def plots(f):
    if f.empty:
        e = empty_fig("No data")
        return e, e, e, e
    if "landscape_class" in f:
        d = f.landscape_class.fillna("unknown").value_counts().rename_axis("class").reset_index(name="pairs")
        p1 = px.bar(d, x="class", y="pairs", title="Landscape distribution")
    else: p1 = empty_fig("Landscape distribution")
    if "date_L" in f:
        s = f.date_L.dt.month.value_counts().reindex(range(1,13), fill_value=0)
        p2 = px.bar(pd.DataFrame({"month":range(1,13),"pairs":s.values}), x="month", y="pairs", title="Pairs by month")
    else: p2 = empty_fig("Pairs by month")
    if {"center_lon","center_lat"}.issubset(f.columns):
        d = f.dropna(subset=["center_lon","center_lat"])
        p3 = px.scatter_geo(d, lon="center_lon", lat="center_lat", color="landscape_class" if "landscape_class" in d else None,
                            hover_name="pair_id", title="Spatial coverage") if len(d) else empty_fig("Spatial coverage")
        if len(d): p3.update_geos(fitbounds="locations", visible=True)
    else: p3 = empty_fig("Spatial coverage")
    qcols = [c for c in ["corr_red","corr_nir"] if c in f]
    if qcols:
        m = f[qcols].melt(var_name="metric", value_name="correlation").dropna()
        p4 = px.histogram(m, x="correlation", color="metric", barmode="overlay", nbins=35, title="Correlation quality")
    else: p4 = empty_fig("Correlation quality")
    return p1,p2,p3,p4


def apply_filters(*args):
    *fargs, max_rows = args
    f = filter_df(*fargs)
    ids = f.pair_id.astype(str).tolist()
    shown = ids[:int(max_rows)]
    p1,p2,p3,p4 = plots(f)
    return table_for(f,max_rows), ids, shown, summary(f), p1,p2,p3,p4, gr.update(choices=ids, value=ids[0] if ids else None), f"Showing {min(len(f),int(max_rows)):,} of {len(f):,} filtered rows."


def metadata(pid):
    if not pid: return {}
    row = DF[DF.pair_id == str(pid)]
    if row.empty: return {"error":f"Unknown pair: {pid}"}
    out = {}
    for k,v in row.iloc[0].items():
        if isinstance(v,pd.Timestamp): out[k]=v.strftime("%Y-%m-%d")
        elif isinstance(v,np.integer): out[k]=int(v)
        elif isinstance(v,np.floating): out[k]=None if np.isnan(v) else float(v)
        elif pd.isna(v): out[k]=None
        else: out[k]=v
    return out


def metric_html(m):
    def z(k,d=3,s=""):
        v=m.get(k)
        if v is None:return "—"
        try:return f"{float(v):.{d}f}{s}"
        except:return f"{v}{s}"
    return f'''<div class="metrics">
    <div><b>{m.get('landscape_class','—')}</b><span>Landscape</span></div>
    <div><b>{m.get('state_or_ut','—')}</b><span>State / UT</span></div>
    <div><b>{z('clear_frac',4)}</b><span>Clear fraction</span></div>
    <div><b>{z('corr_red')}</b><span>Red corr</span></div>
    <div><b>{z('corr_nir')}</b><span>NIR corr</span></div>
    <div><b>{z('time_diff_min',1,' min')}</b><span>Time gap</span></div>
    <div><b>{z('water_frac')}</b><span>Water fraction</span></div>
    <div><b>{z('elevation_p90_m',0,' m')}</b><span>P90 elevation</span></div>
    </div>'''


def landcover_plot(m):
    rows=[]
    for k,v in m.items():
        if v is None: continue
        if k.startswith("esri2025_") and k.endswith("_pct"):
            rows.append(("Esri 2025",k[9:-4],float(v)))
        elif k.startswith("wc2021_") and k.endswith("_pct"):
            rows.append(("WorldCover 2021",k[7:-4],float(v)))
        elif k.startswith("dw_") and k.endswith("_pct"):
            rows.append(("Dynamic World",k[3:-4],float(v)))
    if not rows: return empty_fig("Land-cover composition")
    d=pd.DataFrame(rows,columns=["source","class","percent"])
    return px.bar(d,x="class",y="percent",color="source",barmode="group",title="Land-cover composition")


def load_pair(pid, mode, stretch):
    if not pid:
        p=placeholder("Select a pair")
        return p,p,p,"",{},empty_fig("Land-cover composition")
    m=metadata(pid)
    try:
        lr,hr=read_pair(str(pid))
        a,b,c=render(lr,hr,mode,stretch),render(hr,hr,mode,stretch),rgb_ref(hr)
    except Exception as e:
        a=b=c=placeholder(f"{type(e).__name__}\n\n{e}")
    return a,b,c,metric_html(m),m,landcover_plot(m)


def pick_table(displayed, evt: gr.SelectData):
    if not displayed:return gr.update()
    i=evt.index[0] if isinstance(evt.index,(tuple,list)) else evt.index
    return displayed[int(i)] if 0<=int(i)<len(displayed) else gr.update()


def random_pair(ids):
    return gr.update() if not ids else np.random.default_rng().choice(ids).item()


def step(ids,current,delta):
    if not ids:return gr.update()
    try:i=ids.index(current)
    except ValueError:i=0
    return ids[(i+delta)%len(ids)]


def export_filtered(ids):
    if not ids: raise gr.Error("No rows to export")
    order={p:i for i,p in enumerate(ids)}
    f=DF[DF.pair_id.isin(ids)].copy()
    f["_o"]=f.pair_id.map(order);f=f.sort_values("_o").drop(columns="_o")
    h=tempfile.NamedTemporaryFile(prefix="filtered_pairs_",suffix=".csv",delete=False);h.close()
    f.to_csv(h.name,index=False)
    return h.name


def audit():
    miss_l=[p for p in DF.pair_id if not (LANDSAT_DIR/f"{p}.tif").is_file()]
    miss_s=[p for p in DF.pair_id if not (SENTINEL_DIR/f"{p}.tif").is_file()]
    return {
        "data_dir":str(DATA_DIR),"manifest":str(MANIFEST),"rows":len(DF),
        "duplicate_pair_ids":int(DF.pair_id.duplicated().sum()),
        "missing_landsat_tiffs":len(miss_l),"missing_sentinel_tiffs":len(miss_s),
        "first_missing_landsat":miss_l[:20],"first_missing_sentinel":miss_s[:20],
        "landscape_counts":DF.landscape_class.value_counts().to_dict() if "landscape_class" in DF else {},
    }

CSS='''
.gradio-container{max-width:1700px!important}.hero{padding:18px 20px;border:1px solid var(--border-color-primary);border-radius:18px;margin-bottom:8px}.hero h1{margin:0 0 4px;font-size:28px}.hero p{margin:0;opacity:.75}.kpis,.metrics{display:grid;grid-template-columns:repeat(6,minmax(120px,1fr));gap:10px}.metrics{grid-template-columns:repeat(4,minmax(140px,1fr))}.kpis>div,.metrics>div{padding:13px;border:1px solid var(--border-color-primary);border-radius:14px;display:flex;flex-direction:column;gap:4px}.kpis b{font-size:22px}.kpis span,.metrics span{font-size:12px;opacity:.7}@media(max-width:900px){.kpis{grid-template-columns:repeat(3,1fr)}.metrics{grid-template-columns:repeat(2,1fr)}}
'''


def build_app():
    ids=DF.pair_id.tolist(); first=ids[0] if ids else None
    dmin=DF.date_L.min().strftime("%Y-%m-%d") if "date_L" in DF and DF.date_L.notna().any() else ""
    dmax=DF.date_L.max().strftime("%Y-%m-%d") if "date_L" in DF and DF.date_L.notna().any() else ""
    with gr.Blocks(title="Satellite SR Pair Explorer",theme=gr.themes.Soft(),css=CSS) as demo:
        gr.HTML(f'<div class="hero"><h1>Satellite SR Pair Explorer</h1><p>Landsat 30 m → Sentinel-2 10 m • {len(DF):,} pairs • <code>{MANIFEST.name}</code></p></div>')
        filtered_state=gr.State(ids); displayed_state=gr.State(ids[:250])
        with gr.Tabs():
            with gr.Tab("Explore & filter"):
                with gr.Row():
                    with gr.Column(scale=1,min_width=300):
                        search=gr.Textbox(label="Search",placeholder="pair id, scene id, tile, state...")
                        landscape=gr.Dropdown(choices("landscape_class"),multiselect=True,label="Landscape class")
                        geo=gr.Dropdown(choices("geographic_stratum"),multiselect=True,label="Geographic stratum")
                        states=gr.Dropdown(choices("state_or_ut"),multiselect=True,label="State / UT")
                        with gr.Accordion("More filters",open=False):
                            priority=gr.Dropdown(choices("priority_region"),multiselect=True,label="Priority region")
                            platforms=gr.Dropdown(choices("platform"),multiselect=True,label="Landsat platform")
                            months=gr.Dropdown([str(i) for i in range(1,13)],multiselect=True,label="Months")
                            with gr.Row():
                                start=gr.Textbox(dmin,label="Start date")
                                end=gr.Textbox(dmax,label="End date")
                            min_clear=gr.Slider(0,1,0,step=.001,label="Minimum clear fraction")
                            max_cl=gr.Slider(0,100,100,step=1,label="Max Landsat cloud %")
                            max_cs=gr.Slider(0,100,100,step=1,label="Max Sentinel cloud %")
                            min_red=gr.Slider(-1,1,-1,step=.01,label="Minimum red correlation")
                            min_nir=gr.Slider(-1,1,-1,step=.01,label="Minimum NIR correlation")
                            max_water=gr.Slider(0,1,1,step=.01,label="Maximum water fraction")
                            max_gap=gr.Slider(0,1440,1440,step=5,label="Maximum time gap (min)")
                        sort=gr.Dropdown(["Oldest first","Newest first","Lowest correlation first","Highest correlation first","Highest clear fraction","Largest time gap"],value="Oldest first",label="Sort")
                        max_rows=gr.Dropdown([100,250,500,1000,5000],value=250,label="Max table rows")
                        with gr.Row():
                            apply=gr.Button("Apply filters",variant="primary")
                            export=gr.Button("Export CSV")
                        export_file=gr.File(label="Filtered manifest")
                    with gr.Column(scale=3):
                        stat=gr.HTML(summary(DF)); info=gr.Markdown(f"Showing {min(len(DF),250):,} of {len(DF):,} rows.")
                        table=gr.Dataframe(value=table_for(DF,250),interactive=False,wrap=True,label="Pairs — click any cell in a row")
                p1,p2,p3,p4=plots(DF)
                with gr.Row(): class_plot=gr.Plot(p1); month_plot=gr.Plot(p2)
                with gr.Row(): map_plot=gr.Plot(p3); quality_plot=gr.Plot(p4)
            with gr.Tab("Pair viewer"):
                with gr.Row():
                    selector=gr.Dropdown(ids,value=first,label="Pair ID",allow_custom_value=True,scale=5)
                    prev=gr.Button("← Previous"); rnd=gr.Button("Random"); nxt=gr.Button("Next →")
                with gr.Row():
                    mode=gr.Radio(["RGB","False color (NIR-R-G)","NIR","NDVI"],value="RGB",label="View mode")
                    stretch=gr.Radio(["Auto 2–98%","Fixed 0–3000","Full 0–10000"],value="Auto 2–98%",label="Stretch")
                metrics=gr.HTML()
                with gr.Row():
                    lr=gr.Image(label="Landsat 30 m • 160×160",type="pil",height=430)
                    hr=gr.Image(label="Sentinel-2 10 m • 480×480",type="pil",height=430)
                    ref=gr.Image(label="Sentinel-2 RGB reference",type="pil",height=430)
                with gr.Row(): lc=gr.Plot(); meta=gr.JSON(label="Complete manifest row")
            with gr.Tab("Dataset audit"):
                gr.Markdown("Checks manifest/TIFF consistency without changing any data.")
                audit_btn=gr.Button("Run integrity audit",variant="primary")
                audit_out=gr.JSON(label="Audit report")

        inputs=[search,landscape,geo,priority,states,platforms,months,start,end,min_clear,max_cl,max_cs,min_red,min_nir,max_water,max_gap,sort,max_rows]
        outputs=[table,filtered_state,displayed_state,stat,class_plot,month_plot,map_plot,quality_plot,selector,info]
        apply.click(apply_filters,inputs,outputs); search.submit(apply_filters,inputs,outputs)
        export.click(export_filtered,[filtered_state],[export_file])
        table.select(pick_table,[displayed_state],[selector])
        prev.click(lambda x,y:step(x,y,-1),[filtered_state,selector],[selector])
        nxt.click(lambda x,y:step(x,y,1),[filtered_state,selector],[selector])
        rnd.click(random_pair,[filtered_state],[selector])
        pair_outputs=[lr,hr,ref,metrics,meta,lc]
        selector.change(load_pair,[selector,mode,stretch],pair_outputs)
        mode.change(load_pair,[selector,mode,stretch],pair_outputs)
        stretch.change(load_pair,[selector,mode,stretch],pair_outputs)
        audit_btn.click(audit,outputs=[audit_out])
        demo.load(load_pair,[selector,mode,stretch],pair_outputs)
    return demo



In [ ]:

app=build_app()
share=os.environ.get("GRADIO_SHARE","1").lower() not in {"0","false","no","off"}
port=int(os.environ.get("GRADIO_PORT","7860"))
app.queue(default_concurrency_limit=8).launch(server_name="0.0.0.0",server_port=port,share=share,show_error=True)
